# PersonaPlex sales agent - browser and phone

Runs the agent directly from this notebook. No VS Code, no repo clone: the two app
files are written by Cells 6 and 7, so this notebook is self-contained.

**The Twilio call handling is a port of [chaosloth/spike-persona-plex](https://github.com/chaosloth/spike-persona-plex)** -
the reference app from Twilio's "Build Real-Time Speech to Speech with Twilio Media
Streams and NVIDIA PersonaPlex". Everything that repo does in TypeScript against a
PersonaPlex server on RunPod, `bridge.py` now does in Python against the local
moshi.cpp binary, file for file:

| spike-persona-plex | here |
|---|---|
| `src/index.ts` | `Bridge.media_stream_ws` |
| `src/server.ts` | `twiml_*` builders, `/voice/*` routes |
| `src/bridge-session.ts` | `BridgeSession` |
| `src/call-registry.ts` | `CallRegistry` |
| `src/monitor-ws.ts` | `Bridge.monitor_ws` |
| `src/api-routes.ts` | `/api/calls`, `/api/config` |
| `src/conference-session.ts` | conference helpers, `cross_wire` |
| `src/conference-handler.ts` | `/voice/conference*` |
| `src/outbound-call.ts` | `dial_party_b`, `hangup_call` |
| `src/audio-transcoder.ts` | mu-law tables, `Up8to24` / `Down24to8` |
| `src/personaplex-client.ts` | `Engine` (a process, not a socket) |
| `client/` (React) | `index.html`, one page, no build step |

What that buys over the old `/twilio` endpoint: **inbound** calls to your Twilio
number, a **browser call placed through Twilio** with the Voice SDK, **two-party
conference translation** with the audio cross-wired, a **call registry** so more than
one call can be live at once, and a dashboard that lists calls with per-call metrics
and transcript instead of one global meter. The old high-quality direct-browser path
is still there - see *Cell 11*.

Weights are the **q8_0** build of PersonaPlex 7B
([alvanalrakib/personaplex-7b-q8-GGUF](https://huggingface.co/alvanalrakib/personaplex-7b-q8-GGUF)),
not the q4_k one the moshi.cpp release ships with - about 9 GB to download and
noticeably heavier on a T4. See Cell 5.

## Before you run anything

1. **Settings -> Accelerator -> GPU T4** (one card; the q8_0 weights need about
   9 GB, ~11.5 GB with a full context, of the T4's 14.8 GB)
2. **Settings -> Internet -> On**
3. **Add-ons -> Secrets**:

| Secret | Needed for | Where it comes from |
|---|---|---|
| `NGROK_TOKEN` | everything | dashboard.ngrok.com |
| `TWILIO_ACCOUNT_SID` | any phone call | Twilio console, starts `AC` |
| `TWILIO_AUTH_TOKEN` | any phone call | Twilio console |
| `TWILIO_FROM_NUMBER` | any phone call | your Twilio number, `+1...` |
| `TWILIO_TO_NUMBER` | *Ring my phone* | your verified number, `+91...` |
| `TWILIO_API_KEY_SID` | browser-via-Twilio | Account -> API keys, starts `SK` |
| `TWILIO_API_KEY_SECRET` | browser-via-Twilio | shown once, when you create the key |
| `TWILIO_TWIML_APP_SID` | browser-via-Twilio | Voice -> TwiML Apps, starts `AP` |
| `PARTY_B_PHONE` | conference mode | the other party, `+91...` |

Only the first four matter for a normal phone call. Leave every Twilio secret out and
the browser path still works on its own; the dashboard greys out what it can't do and
says which secret is missing.

**Numbers must be E.164** - a leading `+` and country code, no spaces or dashes.
On a Twilio trial you can only dial numbers you've verified, and Twilio plays its
own trial notice before connecting you.

## Cell 1 - Check the session

In [1]:
import subprocess, sys

gpu = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "none - set Accelerator to GPU T4")
print("Python:", sys.version.split()[0])
net = subprocess.run(["curl","-sI","-m","8","https://github.com"], capture_output=True).returncode == 0
print("Internet:", net)
assert net, "Turn on Internet in the notebook sidebar"

GPU: Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
Python: 3.12.13
Internet: True


## Cell 2 - Paths and secrets

Twilio credentials go into the environment rather than onto a command line, so they
never appear in `ps` output or in this notebook's saved output. `bridge.py` reads them
from there at startup - the same names the repo's `.env.example` uses, plus this
notebook's original ones, so either spelling works.

In [2]:
import os, pathlib
from kaggle_secrets import UserSecretsClient

APP      = "/kaggle/working/agent"
RUNTIME  = "/kaggle/temp/pplex-runtime"
APP_PORT = 8998
LOG      = "/kaggle/temp/agent.log"

os.environ["PPLEX_HOME"] = RUNTIME
for d in (APP, RUNTIME):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

PROMPT = '''You work for Piyush Academy which is an online education institute and your name is Riya.
Information: The Basic Computer Course is four thousand rupees over two months and covers Windows, MS Office, typing and internet basics, and it suits complete beginners and people who want office work.
Tally with GST is seven thousand rupees over three months and covers accounting entries, GST returns and payroll, and it suits people who want accounts or shop billing jobs. 
Full Stack Web Development is twenty five thousand rupees over six months and covers HTML, CSS, JavaScript, React, Node and databases, and it suits people who want developer jobs. 
Classes are live online with about fifteen students in a batch, one hour on weekdays, with a morning batch at seven and evening batches at seven and eight thirty.
Recordings are provided if you miss a class, and there are doubt clearing sessions every Saturday. A free demo class is available before you enroll, you get a certificate on completion, and there is placement assistance with interview practice for the Tally and Full Stack courses.
Fees can be paid in two or three instalments, there is twenty percent off this month, and fees are refundable in the first week. 
You need a laptop and an internet connection, and no prior experience for the Basic or Tally courses. Ask what they want to learn, what they are doing now, and how much time they can give each day, then recommend one course and offer the free demo class.
Keep your answers to one or two short sentences and let them speak. 
only give limited information to user and let user decide most of the things, speak less listen more.
If you do not know something, say you will have the team confirm it. be always polite and calm '''

_sec = UserSecretsClient()
def secret(*names):
    for n in names:
        try:
            v = _sec.get_secret(n)
            if v:
                return v.strip()
        except Exception:
            pass
    return None

NGROK_TOKEN = secret("NGROK_TOKEN", "NGROK_TOKEN_CODE")
assert NGROK_TOKEN, "Add NGROK_TOKEN under Add-ons -> Secrets"

# What each group unlocks. bridge.py reads these straight out of the environment
# (twilio_env() accepts either the repo's names or this notebook's originals).
CORE     = ["TWILIO_ACCOUNT_SID", "TWILIO_AUTH_TOKEN", "TWILIO_FROM_NUMBER"]
DIAL_OUT = ["TWILIO_TO_NUMBER"]
WEBRTC   = ["TWILIO_API_KEY_SID", "TWILIO_API_KEY_SECRET", "TWILIO_TWIML_APP_SID"]
CONF     = ["PARTY_B_PHONE"]

found = []
for k in CORE + DIAL_OUT + WEBRTC + CONF:
    v = secret(k)
    if v:
        os.environ[k] = v
        found.append(k)

# The repo calls the caller ID TWILIO_PHONE_NUMBER; mirror it so either name works.
if "TWILIO_FROM_NUMBER" in found:
    os.environ.setdefault("TWILIO_PHONE_NUMBER", os.environ["TWILIO_FROM_NUMBER"])

print("app files :", APP)
print("runtime   :", RUNTIME)
print("ngrok     : set")

def report(label, keys, note):
    missing = [k for k in keys if k not in found]
    if missing:
        print(f"{label:<10}: off - missing {', '.join(missing)}")
    else:
        print(f"{label:<10}: ready   ({note})")

report("inbound", CORE, "call your Twilio number")
report("dial out", CORE + DIAL_OUT, "Ring my phone")
report("browser tw", CORE + WEBRTC, "browser call routed through Twilio")
report("conference", CORE + CONF, "two-party translated call")
print("browser   : ready   (direct 24 kHz path, no Twilio needed)")

for k in ("TWILIO_FROM_NUMBER", "TWILIO_TO_NUMBER", "PARTY_B_PHONE"):
    n = os.environ.get(k)
    if n:
        ok = n.startswith("+") and n[1:].isdigit()
        print(f"  {k:<20} ...{n[-4:]}  [{'ok' if ok else 'NOT E.164 - needs +countrycode'}]")

app files : /kaggle/working/agent
runtime   : /kaggle/temp/pplex-runtime
ngrok     : set
inbound   : ready   (call your Twilio number)
dial out  : ready   (Ring my phone)
browser tw: off - missing TWILIO_API_KEY_SID, TWILIO_API_KEY_SECRET, TWILIO_TWIML_APP_SID
conference: off - missing PARTY_B_PHONE
browser   : ready   (direct 24 kHz path, no Twilio needed)
  TWILIO_FROM_NUMBER   ...2638  [ok]
  TWILIO_TO_NUMBER     ...7262  [ok]


## Cell 3 - System dependencies

`libsdl2` is what the model binary links against for audio; `ffmpeg` brings the libav*
libraries it needs. The check at the end is the one that matters: the agent reaches
your browser through SDL's **disk** audio driver, which is a compile-time option. If it
were missing the call would never connect, so it's worth knowing before downloading
nine gigabytes.

In [3]:
import subprocess, ctypes, ctypes.util

APT = ["aria2", "libsdl2-2.0-0", "xz-utils", "ffmpeg"]
missing = [p for p in APT
           if subprocess.run(f"dpkg -s {p}", shell=True, capture_output=True).returncode != 0]
if missing:
    pkgs = " ".join(missing)
    print("installing:", pkgs)
    !apt-get -qq update
    !apt-get -qq install -y {pkgs} > /dev/null
else:
    print("apt packages already present")

!pip install -q aiohttp numpy pyngrok twilio

lib = ctypes.util.find_library("SDL2") or "libSDL2-2.0.so.0"
sdl = ctypes.CDLL(lib)
sdl.SDL_GetNumAudioDrivers.restype = ctypes.c_int
sdl.SDL_GetAudioDriver.restype = ctypes.c_char_p
sdl.SDL_GetAudioDriver.argtypes = [ctypes.c_int]
drivers = [sdl.SDL_GetAudioDriver(i).decode() for i in range(sdl.SDL_GetNumAudioDrivers())]
print("\nSDL audio drivers:", ", ".join(drivers))
assert "disk" in drivers, "This libSDL2 has no disk driver; see troubleshooting at the end."
print("disk driver present")

installing: aria2
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 80.5 MB/s eta 0:00:00

SDL audio drivers: pulseaudio, alsa, sndio, pipewire, dsp, disk, dummy
disk driver present


## Cell 4 - The moshi.cpp binary

Prebuilt release, about 160 MB. Skipped if it's already there.

In [4]:
import glob, json, os, subprocess, urllib.request

def find_binary():
    hits = [p for p in glob.glob(f"{RUNTIME}/**/personaplex", recursive=True)
            if os.path.isfile(p)]
    return hits[0] if hits else None

if find_binary():
    print("already present")
else:
    rel = json.load(urllib.request.urlopen(
        "https://api.github.com/repos/Codes4Fun/moshi.cpp/releases/latest"))
    exts = (".tar.xz", ".tar.gz", ".tgz", ".tar.bz2", ".zip")
    linux = [a for a in rel["assets"]
             if "linux" in a["name"].lower() and a["name"].lower().endswith(exts)]
    assert linux, "no Linux asset in release " + rel["tag_name"]
    asset, name = linux[0], linux[0]["name"]
    print(f"{rel['tag_name']} -> {name} ({asset['size']/1e6:.0f} MB)")

    !cd {RUNTIME} && curl -fL --retry 3 -o "{name}" "{asset['browser_download_url']}"

    low = name.lower()
    cmd = (f'unzip -oq "{name}"'  if low.endswith(".zip")     else
           f'tar -xJf "{name}"'   if low.endswith(".tar.xz")  else
           f'tar -xjf "{name}"'   if low.endswith(".tar.bz2") else
           f'tar -xzf "{name}"')
    r = subprocess.run(cmd, shell=True, cwd=RUNTIME, capture_output=True, text=True)
    assert r.returncode == 0, r.stderr
    os.remove(os.path.join(RUNTIME, name))

PPX = find_binary()
assert PPX, "extracted, but no personaplex executable inside"
os.chmod(PPX, 0o755)
BINDIR = os.path.dirname(PPX)

ldd = subprocess.run(["ldd", PPX], capture_output=True, text=True).stdout
gone = [l.strip() for l in ldd.split("\n") if "not found" in l]
assert not gone, "missing shared libraries:\n  " + "\n  ".join(gone)
print("binary:", PPX)

v0.8.0-beta -> moshi-bin-linux-x64-v0.8.0-beta.tar.xz (161 MB)
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  153M  100  153M    0     0  13.2M      0  0:00:11  0:00:11 --:--:-- 13.4M
binary: /kaggle/temp/pplex-runtime/moshi-bin-linux-x64-v0.8.0-beta/personaplex


## Cell 5 - Weights (~9 GB)

The long one. This writes an aria2 manifest for **alvanalrakib/personaplex-7b-q8-GGUF**
and then checks for exactly the 22 files it names - `model-q8_0.gguf`, its config, 18
voice embeddings under `voices/`, and the mimi codec and tokenizer under
`moshi-common/` - so a failure names what's actually missing. Re-running only fetches
what's absent.

The two `moshi-common` files are byte-identical to the ones the q4 build used, so if you
ran the q4 version of this notebook they're skipped.

The folder lands at `Codes4Fun/personaplex-7b-q8-GGUF`, not under `alvanalrakib/`,
because the config inside it reaches for `../moshi-common` - it has to sit beside that
folder. `MODEL_DIR` is what Cells 8 and 9 hand to the binary with `-m`; `CONTEXT` is the
one knob to turn if VRAM gets tight.

In [5]:
import pathlib, shutil

# The model. q8_0 rather than the q4_k the moshi.cpp release defaults to:
#   https://huggingface.co/alvanalrakib/personaplex-7b-q8-GGUF
# personaplex resolves its weights by directory, reading personaplex-config.json
# inside it for the actual filenames, so switching quantisation is a matter of
# pointing -m somewhere else - nothing in the binary needs to change.
HF_REPO    = "alvanalrakib/personaplex-7b-q8-GGUF"
HF_REV     = "11d5fcd0f64cb6d048375ba953812f2009fa1727"   # pinned to "Final upload"
MODEL_GGUF = "model-q8_0.gguf"                            # 8.9 GB
MODEL_DIR  = "Codes4Fun/personaplex-7b-q8-GGUF"           # relative to BINDIR, passed as -m
CONTEXT    = 3000        # frames, 12.5 per second, max 3000. Lower this first if VRAM is tight.

# MODEL_DIR keeps the Codes4Fun/ parent on purpose: the config in this repo names the
# codec as ../moshi-common/mimi-e351c8d8-125.gguf, so the folder has to be a sibling of
# Codes4Fun/moshi-common or the mimi file won't be found.
COMMON = "Codes4Fun/moshi-common"

VOICES = ["NATF0", "NATF1", "NATF2", "NATF3",
          "NATM0", "NATM1", "NATM2", "NATM3",
          "VARF0", "VARF1", "VARF2", "VARF3", "VARF4",
          "VARM0", "VARM1", "VARM2", "VARM3", "VARM4"]

MODEL_URL  = f"https://huggingface.co/{HF_REPO}/resolve/{HF_REV}"
COMMON_URL = ("https://huggingface.co/Codes4Fun/moshi-common/resolve/"
              "a39d64307b971321140a67dc1bc4a8b0e43e4e6b")

wanted, entries = [], []
def add(url, out, sha=None):
    wanted.append(out)
    lines = [url, f"  out={out}", "  auto-file-renaming=false", "  allow-overwrite=false"]
    if sha:                                  # only the shared files publish one
        lines += [f"  checksum=sha-256={sha}", "  check-integrity=true"]
    entries.append("\n".join(lines))

add(f"{MODEL_URL}/{MODEL_GGUF}",            f"{MODEL_DIR}/{MODEL_GGUF}")
add(f"{MODEL_URL}/personaplex-config.json", f"{MODEL_DIR}/personaplex-config.json")
for v in VOICES:
    add(f"{MODEL_URL}/voices/{v}.gguf", f"{MODEL_DIR}/voices/{v}.gguf")
add(f"{COMMON_URL}/tokenizer_spm_32k_3.model", f"{COMMON}/tokenizer_spm_32k_3.model",
    "78d4336533ddc26f9acf7250d7fb83492152196c6ea4212c841df76933f18d2d")
add(f"{COMMON_URL}/mimi-e351c8d8-125.gguf", f"{COMMON}/mimi-e351c8d8-125.gguf",
    "7e0c9ced83cbd035f70b82f1c5602673083fcccec006ea29f48d2e32c60ec697")

MANIFEST = "alvanalrakib_personaplex-7b-q8-GGUF.txt"
(pathlib.Path(BINDIR) / MANIFEST).write_text("\n".join(entries) + "\n")

def survey():
    """What's absent, and what aria2 started but never finished - it leaves a
    .aria2 control file beside anything still in flight."""
    gap = [w for w in wanted if not (pathlib.Path(BINDIR) / w).is_file()]
    part = [str(p.relative_to(BINDIR))[:-6] for p in pathlib.Path(BINDIR).glob("**/*.aria2")]
    return gap, part

gap, part = survey()
if not gap and not part:
    print(f"all {len(wanted)} files already present")
else:
    free = shutil.disk_usage(BINDIR).free / 1e9
    print(f"fetching {len(gap) + len(part)} of {len(wanted)} files, about 9.0 GB; {free:.0f} GB free here")
    if free < 11:
        print("warning: that's tight. Delete the old q4 weights if they're still around.")
    !cd {BINDIR} && aria2c --disable-ipv6 --continue=true --max-tries=5 --retry-wait=5 --max-connection-per-server=4 --split=4 --min-split-size=64M --summary-interval=0 --console-log-level=warn -i {MANIFEST}
    gap, part = survey()

if gap or part:
    print(f"\nstill incomplete - {len(gap)} missing, {len(part)} half-downloaded:")
    for m in (gap + part)[:12]:
        print("   ", m)
    !df -h /kaggle/temp | tail -1
    raise SystemExit("Incomplete. Rerun this cell - aria2 resumes where it stopped.")

!du -sh {BINDIR}/{MODEL_DIR}
print(f"model ready - {len(wanted)} files, {MODEL_DIR}/{MODEL_GGUF}, context {CONTEXT}")

old_q4 = pathlib.Path(BINDIR, "Codes4Fun/personaplex-7b-v1-q4_k-GGUF", "model-q4_k.gguf")
if old_q4.is_file():
    print(f"\nthe old q4 file is still on disk ({old_q4.stat().st_size / 1e9:.1f} GB)")
    print(f"remove it with:  !rm '{old_q4}'")

fetching 22 of 22 files, about 9.0 GB; 1102 GB free here
[#910655 8.1GiB/8.2GiB(98%) CN:3 DL:26MiB ETA:3s]mmmm#b562c8 331MiB/331MiB(100%)] [Checksum:#2.3Mim[#b5
Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
8af11f|OK  |   100KiB/s|/kaggle/temp/pplex-runtime/moshi-bin-linux-x64-v0.8.0-beta/Codes4Fun/personaplex-7b-q8-GGUF/personaplex-config.json
a87f43|OK  |   672KiB/s|/kaggle/temp/pplex-runtime/moshi-bin-linux-x64-v0.8.0-beta/Codes4Fun/personaplex-7b-q8-GGUF/voices/NATF1.gguf
f89b9d|OK  |   907KiB/s|/kaggle/temp/pplex-runtime/moshi-bin-linux-x64-v0.8.0-beta/Codes4Fun/personaplex-7b-q8-GGUF/voices/NATF3.gguf
d68ae5|OK  |   173KiB/s|/kaggle/temp/pplex-runtime/moshi-bin-linux-x64-v0.8.0-beta/Codes4Fun/personaplex-7b-q8-GGUF/voices/NATF0.gguf
29bd59|OK  |   160KiB/s|/kaggle/temp/pplex-runtime/moshi-bin-linux-x64-v0.8.0-beta/Codes4Fun/personaplex-7b-q8-GGUF/voices/NATF2.gguf
439f1e|OK  |   244KiB/s|/kaggle/

## Cell 6 - The server

`bridge.py`. Twilio Media Streams on `/media-stream`, TwiML on `/voice/*`, the call
registry and its monitor socket on `/ws/monitor`, the REST-ish API on `/api/*`, and
the direct browser audio path on `/ws`. Writing it as a file rather than running it
inline keeps it restartable without re-running the cell.

Roughly 2,200 lines, and about a third of it is the audio work this notebook already
had - the mu-law tables, the Kaiser resamplers, the echo guard and the WSOLA
time-stretch. The call handling above it is the port described in Cell 0.

In [6]:
%%writefile /kaggle/working/agent/bridge.py
#!/usr/bin/env python3
"""
bridge.py - Twilio Media Streams in front of moshi.cpp's `personaplex` binary.

Call handling is a port of chaosloth/spike-persona-plex (the Twilio blog post
"Build Real-Time Speech to Speech with Twilio Media Streams and NVIDIA
PersonaPlex"). Everything that repo does in TypeScript against a remote
PersonaPlex server, this does in Python against the local binary:

    twilio-project/src/index.ts               -> BRIDGE ROUTES + media_stream_ws
    twilio-project/src/server.ts              -> twiml_* builders, /voice/* routes
    twilio-project/src/bridge-session.ts      -> BridgeSession
    twilio-project/src/call-registry.ts       -> CallRegistry
    twilio-project/src/monitor-ws.ts          -> monitor_ws + handle_client_message
    twilio-project/src/api-routes.ts          -> /api/calls, /api/config
    twilio-project/src/conference-session.ts  -> conference helpers
    twilio-project/src/conference-handler.ts  -> /voice/conference*
    twilio-project/src/outbound-call.ts       -> dial_party_b, hangup_call
    twilio-project/src/audio-transcoder.ts    -> mu-law tables + Down24to8/Up8to24
    twilio-project/src/audio-buffer.ts        -> not needed (see below)
    twilio-project/src/personaplex-client.ts  -> Engine (local process, not a socket)
    twilio-project/client/                    -> index.html

ONE DELIBERATE DIVERGENCE. The repo wraps caller audio in Ogg Opus because its
PersonaPlex lives on a RunPod GPU behind a websocket that only accepts Ogg. Ours
is a process on this machine reading float32 straight off a pipe, so the Opus
round trip would be a lossy encode and an immediate decode for no reason. The
sample-rate conversion the repo does with a two-tap linear interpolator is kept
as the Kaiser-windowed filter this notebook already had; that is also what makes
AudioBuffer unnecessary, since nothing downstream needs fixed 480-sample frames.
Every other part of the call path - route names, TwiML shape, <Parameter>
passing, session lifecycle, registry events, metrics, conference cross-wiring -
follows the repo.

The engine itself is a desktop app: it talks to a sound card through SDL. A
Kaggle box has no sound card, so we point SDL at its `disk` driver and hand it
two named pipes where it expects a device:

    caller/mic  --ws-->  [pipe]  -->  SDL capture   -->  personaplex
    caller/spkr <--ws--  [pipe]  <--  SDL playback  <--  personaplex

Both pipes carry float32 mono @ 24 kHz, which is exactly what personaplex asks
SDL for (want.freq=24000, want.format=AUDIO_F32, want.channels=1).

Pacing: Python is the clock. We read the playback pipe on a wall-clock budget of
24000*4 bytes/sec and no faster, so when the model outruns us the pipe fills, SDL's
write blocks, and the model throttles itself. Caller audio arrives in real time
already; if the model falls behind we drop the oldest buffered audio rather than
let latency grow without bound.
"""

import argparse
import asyncio
from collections import deque
import base64
import errno
import fcntl
import json
import math
import os
import shutil
import signal
import sys
import time
import uuid
from pathlib import Path
from xml.sax.saxutils import escape as xml_escape, quoteattr as xml_quoteattr

import numpy as np
from aiohttp import web, WSMsgType

SAMPLE_RATE = 24000
FRAME_SAMPLES = 1920            # 80 ms, one mimi frame at 12.5 fps
FRAME_BYTES_F32 = FRAME_SAMPLES * 4
BYTES_PER_SEC = SAMPLE_RATE * 4  # float32 mono
TICK = 0.02                      # bridge loop period, seconds
MAX_INPUT_BACKLOG = FRAME_BYTES_F32 * 4   # ~320 ms, then we start dropping
F_SETPIPE_SZ = 1031

# src/bridge-session.ts constants
TWILIO_RATE = 8000
PERSONAPLEX_RATE = SAMPLE_RATE
TWILIO_FRAME = 160               # 20 ms of 8 kHz mu-law, one Twilio media message

VOICES = ["NATF0", "NATF1", "NATF2", "NATF3",
          "NATM0", "NATM1", "NATM2", "NATM3",
          "VARF0", "VARF1", "VARF2", "VARF3", "VARF4",
          "VARM0", "VARM1", "VARM2", "VARM3", "VARM4"]


def log(*a):
    print(f"[bridge {time.strftime('%H:%M:%S')}]", *a, flush=True)


def now_ms() -> int:
    return int(time.time() * 1000)


def f32_to_i16(buf: bytes) -> bytes:
    x = np.frombuffer(buf, dtype="<f4")
    return (np.clip(x, -1.0, 1.0) * 32767.0).astype("<i2").tobytes()


def i16_to_f32(buf: bytes) -> bytes:
    x = np.frombuffer(buf, dtype="<i2").astype("<f4") / 32768.0
    return x.tobytes()


def set_pipe_size(fd: int, size: int):
    try:
        fcntl.fcntl(fd, F_SETPIPE_SZ, size)
    except OSError:
        pass  # not permitted / not supported; defaults are fine


def norm_voice(v: str) -> str:
    """The repo names voices as server-side files ("NATF2.pt"); the local binary
    wants the bare embedding name. Accept either."""
    v = (v or "NATF1").strip()
    return v[:-3] if v.endswith(".pt") else v


# --------------------------------------------------------------------------
# Telephony: Twilio Media Streams carry 8 kHz mu-law, the model speaks 24 kHz
# float32. audioop would do the codec but it was removed in Python 3.13, so
# these are numpy lookup tables, verified bit-exact against the reference codec.
# (src/audio-transcoder.ts MULAW_DECODE_TABLE / linearToMulaw)
# --------------------------------------------------------------------------

MU_BIAS = 0x84
MU_CLIP = 32635


def _build_mulaw_tables():
    # decode: 256 codes -> int16
    u = np.arange(256, dtype=np.int32)
    uc = (~u) & 0xFF
    sign = uc & 0x80
    exponent = (uc >> 4) & 0x07
    mantissa = uc & 0x0F
    mag = (((mantissa << 3) + MU_BIAS) << exponent) - MU_BIAS
    dec = np.where(sign != 0, -mag, mag).astype(np.int16)

    # encode: int16 -> 256 codes.
    # G.711 quantises to 14 bits first; without this we differ from every other
    # telephony stack on 381 boundary values.
    x = (np.arange(-32768, 32768, dtype=np.int32) >> 2) << 2
    s = np.where(x < 0, 0x80, 0)
    a = np.minimum(np.abs(x), MU_CLIP) + MU_BIAS
    # exact integer segment search; np.log2 is off by an ULP at powers of two
    exp = np.zeros_like(a)
    for shift in range(1, 8):
        exp = np.where(a >= (1 << (shift + 7)), shift, exp)
    man = (a >> (exp + 3)) & 0x0F
    enc = ((~(s | (exp << 4) | man)) & 0xFF).astype(np.uint8)
    return enc, dec


_ENC, _DEC = _build_mulaw_tables()


def mulaw_encode(pcm16: np.ndarray) -> bytes:
    idx = np.asarray(pcm16, dtype=np.int32) + 32768
    return _ENC[idx].tobytes()


def mulaw_decode(data: bytes) -> np.ndarray:
    return _DEC[np.frombuffer(data, dtype=np.uint8)]


def _lowpass(cutoff, sr, taps=127, beta=7.0):
    """Kaiser-windowed sinc. 127 taps at beta 7 buys roughly 70 dB of stopband,
    which is what stops a 6 kHz tone folding down to 2 kHz on the way to 8 kHz."""
    n = np.arange(taps) - (taps - 1) / 2
    h = np.sinc(2 * cutoff / sr * n) * np.kaiser(taps, beta)
    return (h / h.sum()).astype(np.float32)


class Down24to8:
    """24 kHz float32 -> 8 kHz float32. Anti-aliased, then every third sample.

    Cutoff sits at 3500 rather than 3400 with a longer filter, so 2800-3400 Hz
    comes through flat instead of rolling off 6 dB. That band carries the
    consonants, which is most of what intelligibility is on a phone. Still 105 dB
    down at 4 kHz, so nothing folds back.
    """

    def __init__(self):
        self.h = _lowpass(3500, 24000, taps=201)
        self.z = np.zeros(len(self.h) - 1, dtype=np.float32)
        self.phase = 0

    def __call__(self, x):
        x = np.asarray(x, dtype=np.float32)
        buf = np.concatenate([self.z, x])
        y = np.convolve(buf, self.h, "valid")
        out = y[self.phase::3]
        consumed = len(y)
        self.phase = (self.phase - consumed) % 3
        self.z = buf[-(len(self.h) - 1):] if len(buf) >= len(self.h) - 1 else buf
        return out.astype(np.float32)


class Up8to24:
    """8 kHz float32 -> 24 kHz float32. Zero-stuff then interpolate."""

    def __init__(self):
        self.h = _lowpass(3400, 24000) * 3.0
        self.z = np.zeros(len(self.h) - 1, dtype=np.float32)

    def __call__(self, x):
        x = np.asarray(x, dtype=np.float32)
        up = np.zeros(len(x) * 3, dtype=np.float32)
        up[::3] = x
        buf = np.concatenate([self.z, up])
        y = np.convolve(buf, self.h, "valid")
        self.z = buf[-(len(self.h) - 1):]
        return y.astype(np.float32)


def _highpass_taps(cutoff, sr, taps=127, beta=7.0):
    h = -_lowpass(cutoff, sr, taps, beta)
    h[(taps - 1) // 2] += 1.0
    return h.astype(np.float32)


class _FIR:
    """Stateful FIR so chunk boundaries don't click."""

    def __init__(self, taps):
        self.h = taps
        self.z = np.zeros(len(taps) - 1, dtype=np.float32)

    def __call__(self, x):
        buf = np.concatenate([self.z, x.astype(np.float32)])
        y = np.convolve(buf, self.h, "valid")
        self.z = buf[-(len(self.h) - 1):]
        return y.astype(np.float32)


def _rms(x):
    return float(np.sqrt(np.mean(x * x))) if len(x) else 0.0


class EchoGuard:
    """Duck the caller's channel while the agent is the one talking.

    We hold a decaying memory of how loud we have been sending, which covers the
    few hundred ms it takes our audio to reach the handset and come back. While
    that memory is warm, anything arriving that is consistent with an echo gets
    attenuated. Anything clearly louder than the echo we would expect is real
    speech and passes, so the caller can still interrupt.
    """

    def __init__(self, far_decay=0.94, duck=0.03, erl=0.06):
        self.far = 0.0
        self.far_decay = far_decay
        self.erl = erl              # echo return loss: how much comes back
        self.duck = duck
        self.gain = 1.0
        self.ducking = False

    def note_far(self, pcm):
        """Called with whatever we just sent to the caller."""
        self.far = max(_rms(pcm), self.far * self.far_decay)

    def __call__(self, near_rms):
        if self.far < 0.004:
            target = 1.0
        else:
            expected_echo = self.far * self.erl
            if near_rms < expected_echo * 3.0:
                # consistent with echo: duck, and refine what the return loss is
                obs = min(1.0, near_rms / max(self.far, 1e-9))
                self.erl = float(np.clip(0.98 * self.erl + 0.02 * obs, 0.005, 0.3))
                target = self.duck
            else:
                target = 1.0                      # double talk, let them through
        # open fast so a real interruption isn't clipped, close slower
        a = 0.5 if target > self.gain else 0.25
        self.gain += a * (target - self.gain)
        self.ducking = self.gain < 0.5
        return self.gain


class BandwidthExtend:
    """Phone audio stops dead at 3.4 kHz. The model was trained on 24 kHz speech,
    so that empty top half is unlike anything it heard in training. This
    translates the speech band up to sit above the gap, well under the level of
    the voice itself, so the spectrum looks less alien without colouring her."""

    def __init__(self, sr=SAMPLE_RATE, shift=4200, mix_db=-12.0):
        self.sr = sr
        self.shift = shift
        self.mix = 10 ** (mix_db / 20)
        self.phase = 0
        self.hp = _FIR(_highpass_taps(4000, sr))

    def __call__(self, x):
        n = len(x)
        t = (np.arange(n) + self.phase) / self.sr
        self.phase = (self.phase + n) % self.sr        # keep the carrier continuous
        hi = self.hp(x * np.cos(2 * np.pi * self.shift * t).astype(np.float32))
        rh, rx = _rms(hi), _rms(x)
        if rh > 1e-9 and rx > 1e-6:
            hi = hi * (rx * self.mix / rh)
        return (x + hi).astype(np.float32)


class CallerAudio:
    """Everything applied to inbound caller audio, in order."""

    def __init__(self, sr=SAMPLE_RATE, target_rms=0.10, max_gain_db=24.0,
                 frame=480, bwe=False):
        self.sr = sr
        self.bwe = BandwidthExtend(sr) if bwe else None
        self.target = target_rms
        self.max_gain = 10 ** (max_gain_db / 20)
        self.rumble = _FIR(_highpass_taps(90, sr))
        self.echo = EchoGuard()
        self.hist = deque(maxlen=150)      # ~3 s of frame levels
        self.noise = 0.003
        self.env = 0.0
        self.gain = 1.0
        self.gate = 1.0
        self.last_in = 0.0
        self.last_out = 0.0

    def note_far(self, pcm):
        self.echo.note_far(pcm)

    def __call__(self, x):
        x = self.rumble(np.asarray(x, dtype=np.float32))
        rms = _rms(x)

        # 1. echo first, so the gain stages never see her own voice
        x = x * self.echo(rms)
        rms = _rms(x)

        # 2. noise floor from the quietest tenth of the last three seconds
        self.hist.append(rms)
        if len(self.hist) >= 25:
            self.noise = max(float(np.percentile(self.hist, 10)), 1e-5)

        # 3. gate: keep the floor down instead of amplifying it into speech
        speech = rms > self.noise * 4.0
        target_gate = 1.0 if speech else (0.15 if rms > self.noise * 2.0 else 0.02)
        a = 0.6 if target_gate > self.gate else 0.08
        self.gate += a * (target_gate - self.gate)
        x = x * self.gate

        # 4. gain, adapted only on real speech
        if speech:
            k = 0.30 if rms > self.env else 0.05
            self.env += k * (rms - self.env)
        want = min(self.max_gain, self.target / self.env) if self.env > 1e-5 else self.gain
        self.gain += 0.08 * (want - self.gain)
        y = x * self.gain

        if self.bwe is not None and speech:
            y = self.bwe(y)

        peak = float(np.abs(y).max()) if len(y) else 0.0
        if peak > 0.95:
            y = y * (0.95 / peak)

        self.last_in = self.env
        self.last_out = self.env * self.gain if speech or self.gate > 0.5 else 0.0
        return y.astype(np.float32)


class HandsetOut:
    """Outbound conditioning: remove rumble, hold a steady level, limit."""

    def __init__(self, sr=8000, target_rms=0.14, ceiling=0.95):
        # 80 Hz, not 300. The phone network does its own band-limiting; all we
        # need to remove is DC and rumble, which waste mu-law's dynamic range.
        # A 300 Hz corner takes 14 dB out of a female voice's fundamental and
        # makes her sound thin - measured, not guessed.
        self.hp = _FIR(_highpass_taps(80, sr, taps=127))
        self.target = target_rms
        self.ceiling = ceiling
        self.env = 0.0
        self.gain = 1.0

    def __call__(self, x):
        x = self.hp(np.asarray(x, dtype=np.float32))
        rms = _rms(x)
        if rms > 0.004:
            k = 0.3 if rms > self.env else 0.03
            self.env += k * (rms - self.env)
        if self.env > 1e-4:
            want = float(np.clip(self.target / self.env, 0.5, 4.0))
            self.gain += 0.05 * (want - self.gain)
        y = x * self.gain
        peak = float(np.abs(y).max()) if len(y) else 0.0
        if peak > self.ceiling:
            y = y * (self.ceiling / peak)
        return y.astype(np.float32)


class Limiter:
    """Keeps peaks off the mu-law ceiling. Instant attack, slow release, so it
    ducks a loud syllable without pumping the whole utterance."""

    def __init__(self, ceiling=0.95, release=0.05):
        self.ceiling = ceiling
        self.release = release
        self.gain = 1.0

    def __call__(self, x):
        peak = float(np.abs(x).max())
        want = 1.0 if peak < 1e-6 else min(1.0, self.ceiling / peak)
        if want < self.gain:
            self.gain = want                      # attack now
        else:
            self.gain += (want - self.gain) * self.release
        return x * self.gain


class Stretcher:
    """WSOLA: changes duration without changing pitch.

    Speech gets stretched by `rate`; silence passes through untouched, and gets
    compressed when the listener is running a backlog. That last part is what keeps
    the call from drifting - the model still emits audio at exactly real time, so
    the extra duration has to come out of the gaps between her sentences.

    The analysis grid advances by exactly hop*rate regardless of where the
    similarity search lands, so search offsets stay bounded perturbations instead
    of accumulating into drift. The search is also biased toward the grid position,
    which stops it locking onto a neighbouring pitch period.
    """

    def __init__(self, rate=1.0, win=512, search=128, sr=SAMPLE_RATE):
        self.rate = float(rate)
        self.win = win
        self.hop_s = win // 2
        self.search = search
        self.sr = sr
        self.window = np.hanning(win).astype(np.float32)
        # prefer offsets near the grid; sigma ~ half the search range
        d = np.arange(-search, search + 1, dtype=np.float32)
        self.bias = np.exp(-0.5 * (d / (search * 0.6)) ** 2)
        self.buf = np.zeros(0, dtype=np.float32)
        self.tail = np.zeros(self.hop_s, dtype=np.float32)
        self.template = None
        self.ideal = 0.0
        self.in_n = 0
        self.out_n = 0
        # Hysteresis: a word's tail decays through the threshold, and a bare
        # comparison would flip to "silence" and speed the tail up mid-word.
        self.speech_on = 0.010
        self.speech_off = 0.004
        self.in_speech = False
        # Silence is where the stretched time has to be paid back, but paying it
        # back at 3x chops breaths and consonants. 1.6 is inaudible.
        self.max_silence_rate = 1.6
        # Rate is slewed, never stepped. Jumping between 0.75 and 3.0 several
        # times a second is what makes a voice sound robotic.
        self.cur_rate = 1.0
        self.slew = 0.03
        self.backlog_cap = 1.5

    def backlog(self):
        return (self.out_n - self.in_n) / self.sr

    def set_rate(self, rate):
        self.rate = float(rate)

    def _target_rate(self, frame):
        rms = float(np.sqrt(np.mean(frame * frame)))
        if self.in_speech:
            if rms < self.speech_off:
                self.in_speech = False
        elif rms > self.speech_on:
            self.in_speech = True

        b = self.backlog()
        if self.in_speech:
            r = self.rate
            if b > self.backlog_cap:          # too far ahead, ease back toward real time
                r = min(1.0, self.rate + 0.15)
            return r
        return self.max_silence_rate if b > 0.2 else 1.0

    def _local_rate(self, frame):
        target = self._target_rate(frame)
        step = target - self.cur_rate
        self.cur_rate += max(-self.slew, min(self.slew, step))
        return self.cur_rate

    def process(self, x):
        x = np.asarray(x, dtype=np.float32)
        if self.rate == 1.0 and self.backlog() <= 0.25:
            self.in_n += len(x)
            self.out_n += len(x)
            return x

        self.buf = np.concatenate([self.buf, x])
        self.in_n += len(x)
        L, Hs, S = self.win, self.hop_s, self.search
        out = []

        while True:
            nominal = int(round(self.ideal))
            if nominal + L + Hs + S >= len(self.buf):
                break

            # find the frame that continues most smoothly from what we just emitted
            if self.template is None:
                pos = nominal
            else:
                lo = max(0, nominal - S)
                hi = min(len(self.buf) - L - Hs, nominal + S)
                if hi <= lo:
                    pos = nominal
                else:
                    region = self.buf[lo:hi + Hs]
                    corr = np.correlate(region, self.template, "valid")
                    energy = np.sqrt(np.convolve(region * region,
                                                 np.ones(Hs, np.float32),
                                                 "valid")) + 1e-6
                    score = corr / energy
                    b = self.bias[(lo - nominal + S):(hi - nominal + S + 1)]
                    if len(b) == len(score):
                        score = score * b
                    pos = lo + int(np.argmax(score))

            frame = self.buf[pos:pos + L] * self.window
            frame[:Hs] += self.tail
            out.append(frame[:Hs].copy())
            self.tail = frame[Hs:].copy()
            self.template = self.buf[pos + Hs:pos + 2 * Hs].copy()

            # grid advances by the ideal amount, not by where the search landed
            r = self._local_rate(self.buf[pos:pos + L])
            self.ideal += Hs * r
            self.out_n += Hs

            if self.ideal > 4 * L:
                drop = int(self.ideal) - 2 * L
                self.buf = self.buf[drop:]
                self.ideal -= drop

        return np.concatenate(out) if out else np.zeros(0, dtype=np.float32)


# ==========================================================================
# src/types.ts + src/call-registry.ts
#
# One process-wide registry of calls. Everything the dashboard shows comes
# from here, and every mutation emits a monitor event that monitor_ws
# broadcasts. Same event names and payload shapes as the repo, so the client
# reducer is a straight port too.
# ==========================================================================

PRUNE_AFTER_MS = 30 * 60 * 1000

# src/call-registry.ts PROMPT_TEMPLATE - the interpreter persona
def PROMPT_TEMPLATE(lang: str) -> str:
    return (f"You work for TransLingo which is a live interpretation service and "
            f"your name is Maria. You are a {lang} interpreter. Your ONLY job is to "
            f"repeat exactly what the caller says, but in {lang}. Do NOT answer "
            f"questions. Do NOT have a conversation. Do NOT add anything. Just "
            f"translate their exact words into {lang}, nothing else.")


class CallRegistry:
    """Port of src/call-registry.ts. `on_monitor` is the EventEmitter."""

    def __init__(self):
        self.calls = {}                 # callSid -> call dict
        self._listeners = []
        self.config = {
            "mode": os.environ.get("CALL_MODE", "single"),
            "targetLanguage": os.environ.get("TARGET_LANGUAGE", "Spanish"),
            "voicePrompt": os.environ.get("VOICE_PROMPT", "NATF1"),
            "textPrompt": os.environ.get("TEXT_PROMPT", "") or PROMPT_TEMPLATE("Spanish"),
            "partyBPhone": os.environ.get("PARTY_B_PHONE", ""),
            "voicePromptB": os.environ.get("VOICE_PROMPT_B", "NATM1"),
            "textPromptB": os.environ.get("TEXT_PROMPT_B_TO_A", "") or PROMPT_TEMPLATE("English"),
            "targetLanguageB": os.environ.get("TARGET_LANGUAGE_B", "English"),
            # local-engine settings the repo doesn't need (its model is remote and
            # configured at deploy time); these ride along in the same object so
            # the config panel can drive them.
            "temperature": 0.8,
            "context": 3000,
            "rate": 1.0,
            "seed": -1,
            "bwe": False,
        }

    # ---------- events ----------

    def on_monitor(self, fn):
        self._listeners.append(fn)

    def _emit(self, event: dict):
        for fn in list(self._listeners):
            try:
                fn(event)
            except Exception as e:      # a dead socket must not stop the call
                log("monitor listener error:", e)

    # ---------- config ----------

    def get_config(self) -> dict:
        return dict(self.config)

    def update_config(self, partial: dict) -> dict:
        self.config.update(partial or {})
        # src/call-registry.ts: setting a language without an explicit prompt
        # regenerates the prompt from the template.
        if partial.get("targetLanguage") and not partial.get("textPrompt"):
            self.config["textPrompt"] = PROMPT_TEMPLATE(self.config["targetLanguage"])
        self._emit({"type": "config:updated", "config": self.get_config()})
        return self.get_config()

    def engine_settings(self, role: str = "caller-a") -> tuple:
        """(prompt, voice, temperature, context, seed) for one leg of a call.
        Port of buildPersonaPlexConfig in src/conference-session.ts."""
        c = self.config
        if role == "caller-b":
            prompt, voice = c["textPromptB"], c["voicePromptB"]
        else:
            prompt, voice = c["textPrompt"], c["voicePrompt"]
        return (prompt or "", norm_voice(voice),
                float(c.get("temperature", 0.8)), int(c.get("context", 3000)),
                int(c.get("seed", -1)))

    # ---------- calls ----------

    def register_call(self, call_sid, stream_sid, mode, role, conference_name):
        call = {
            "callSid": call_sid,
            "streamSid": stream_sid,
            "mode": mode,
            "role": role,
            "conferenceName": conference_name,
            "status": "active",
            "startTime": now_ms(),
            "endTime": None,
            "duration": 0,
            "transcript": [],
            "notice": "",
            "metrics": {"uptime": 0, "packetsIn": 0, "packetsOut": 0,
                        "latency": 0, "jitter": 0, "fps": 0,
                        "ctxLeft": 0, "ctxTotal": 0, "echo": False},
        }
        self.calls[call_sid] = call
        self._emit({"type": "call:start", "callSid": call_sid, "streamSid": stream_sid,
                    "mode": mode, "role": role, "startTime": call["startTime"]})

    def add_transcript(self, call_sid, speaker, text):
        """Consecutive chunks from the same speaker merge into one entry, exactly
        as the repo does - personaplex emits partial tokens, not sentences."""
        call = self.calls.get(call_sid)
        if not call:
            return
        last = call["transcript"][-1] if call["transcript"] else None
        if last and last["speaker"] == speaker:
            if text:
                skip_space = (not last["text"]) or text[0] in ".!?,;:¿¡ \t\n"
                last["text"] = last["text"] + text if skip_space else last["text"] + " " + text
            self._emit({"type": "call:text", "callSid": call_sid, "speaker": speaker,
                        "text": last["text"], "timestamp": last["timestamp"]})
        else:
            entry = {"speaker": speaker, "text": text, "timestamp": now_ms()}
            call["transcript"].append(entry)
            self._emit({"type": "call:text", "callSid": call_sid, "speaker": speaker,
                        "text": text, "timestamp": entry["timestamp"]})

    def add_notice(self, call_sid, text):
        """An out-of-band message about the call itself, not something anybody
        said - "30 seconds of context left", "ended at the length limit"."""
        call = self.calls.get(call_sid)
        if not call:
            return
        call["notice"] = text
        self._emit({"type": "call:notice", "callSid": call_sid, "text": text})

    def update_metrics(self, call_sid, metrics: dict):
        call = self.calls.get(call_sid)
        if not call:
            return
        call["metrics"].update(metrics)
        call["duration"] = (now_ms() - call["startTime"]) / 1000
        call["metrics"]["uptime"] = call["duration"]
        ev = {"type": "call:metrics", "callSid": call_sid}
        ev.update(call["metrics"])
        self._emit(ev)

    def end_call(self, call_sid, reason):
        call = self.calls.get(call_sid)
        if not call or call["status"] == "ended":
            return
        call["status"] = "ended"
        call["endTime"] = now_ms()
        call["duration"] = (call["endTime"] - call["startTime"]) / 1000
        self._emit({"type": "call:end", "callSid": call_sid,
                    "duration": call["duration"], "reason": reason})

    def get_all_calls(self):
        return list(self.calls.values())

    def get_call(self, call_sid):
        return self.calls.get(call_sid)

    def prune(self):
        cutoff = now_ms() - PRUNE_AFTER_MS
        for sid in [s for s, c in self.calls.items()
                    if c["status"] == "ended" and c["endTime"] and c["endTime"] < cutoff]:
            del self.calls[sid]


call_registry = CallRegistry()


# ==========================================================================
# src/personaplex-client.ts
#
# The repo opens a websocket to a PersonaPlex server and speaks its three-byte
# protocol (0x00 handshake, 0x01 audio, 0x02 text). Ours starts the binary and
# speaks to it over two pipes, but the surface a session sees is the same: a
# thing you start(), feed audio to, and that emits 'audio' / 'text' / 'state'.
# ==========================================================================


class Engine:
    """One personaplex process plus its two pipes."""

    def __init__(self, binary: str, workdir: str, on_event, rate: float = 1.0,
                 model: str = "", name: str = "engine"):
        self.binary = binary
        self.workdir = workdir
        # Which weights to load, as a directory relative to workdir. Empty means
        # let the binary fall back to the q4_k directory it looks for by default.
        self.model = model
        self.name = name
        self.rate = float(rate)
        self.stretch = Stretcher(self.rate)
        # Chrome already levels and echo-cancels the mic, so this is gentler:
        # mostly the rumble filter and a noise gate so room tone never reaches
        # the model as speech.
        self.mic_cond = CallerAudio(target_rms=0.09, max_gain_db=12.0)
        self.out_limiter = Limiter(ceiling=0.97)
        self.on_event = on_event          # called with dict, from the event loop
        self.proc = None
        self.in_fd = None                 # we write caller audio here
        self.out_fd = None                # we read model audio here
        self.in_buf = bytearray()
        self.state = "idle"
        self.settings = None
        self.used = False
        self.owner = None                 # BridgeSession holding this engine
        self.started_at = 0.0
        self.ready = False
        self.out_bytes = 0
        self._fps_window = []
        self._tasks = []
        # Unique per engine: a conference runs two of these side by side and
        # they must not share a fifo.
        self._pipe_dir = Path(workdir) / "_pipes" / name

    # ---------- lifecycle ----------

    async def start(self, prompt, voice, temperature, context, seed=None):
        await self.stop()

        self._pipe_dir.mkdir(parents=True, exist_ok=True)
        mic = self._pipe_dir / "mic.f32"
        spk = self._pipe_dir / "spk.f32"
        for p in (mic, spk):
            if p.exists():
                p.unlink()
            os.mkfifo(p, 0o600)

        env = dict(os.environ)
        env["SDL_AUDIODRIVER"] = "disk"
        env["SDL_DISKAUDIOFILEIN"] = str(mic)   # SDL reads our caller audio from here
        env["SDL_DISKAUDIOFILE"] = str(spk)     # SDL writes model audio to here
        env["SDL_DISKAUDIODELAY"] = "0"         # we do the pacing, not SDL
        env["SDL_VIDEODRIVER"] = "dummy"

        cmd = [self.binary, "-v", norm_voice(voice), "-c", str(int(context)),
               "-t", str(float(temperature))]
        if self.model:
            # personaplex reads personaplex-config.json inside this directory for
            # the weight, codec and tokenizer filenames, and voices/ under it for
            # the embeddings, so this one flag moves the whole model.
            cmd += ["-m", self.model]

        # personaplex prints "ready" with printf and never flushes it; the only
        # fflush(stdout) calls sit inside the generation loop, so "ready" stays
        # stuck in libc's buffer until the model actually says something - which
        # it won't until someone feeds the mic. stdbuf turns that buffering off so
        # readiness is observable the moment it happens.
        if shutil.which("stdbuf"):
            cmd = ["stdbuf", "-o0", "-e0"] + cmd
        else:
            log("WARNING: stdbuf not found; readiness detection may lag")
        if seed is not None and int(seed) >= 0:
            cmd += ["-s", str(int(seed))]
        if prompt and prompt.strip():
            cmd += ["-p", prompt.strip()]

        log(f"[{self.name}] launching:",
            " ".join(repr(c) if " " in c else c for c in cmd))
        self.settings = (prompt or "", norm_voice(voice), float(temperature),
                         int(context), int(seed if seed is not None else -1))
        self.used = False          # has anyone actually talked to this process?
        self.state = "starting"
        self.ready = False
        self.out_bytes = 0
        self._fps_window = []
        self.in_buf = bytearray()
        self.stretch = Stretcher(self.rate)
        self.mic_cond = CallerAudio(target_rms=0.09, max_gain_db=12.0)
        self.out_limiter = Limiter(ceiling=0.97)
        self.started_at = time.monotonic()

        self.proc = await asyncio.create_subprocess_exec(
            *cmd,
            cwd=self.workdir,
            env=env,
            stdout=asyncio.subprocess.PIPE,
            stderr=asyncio.subprocess.STDOUT,
            stdin=asyncio.subprocess.DEVNULL,
        )
        await self._emit({"t": "state", "state": "starting",
                          "msg": "Loading the model. This takes a few seconds."})

        # Read end of a fifo opens immediately with O_NONBLOCK even with no writer.
        self.out_fd = os.open(spk, os.O_RDONLY | os.O_NONBLOCK)
        set_pipe_size(self.out_fd, 65536)

        self._tasks = [
            asyncio.create_task(self._pump_stdout()),
            asyncio.create_task(self._open_mic(mic)),
            asyncio.create_task(self._pump_audio()),
            asyncio.create_task(self._watch_proc()),
        ]

    async def _open_mic(self, path):
        """Write end of a fifo raises ENXIO until the reader (SDL) shows up."""
        deadline = time.monotonic() + 300
        while time.monotonic() < deadline:
            if self.proc is None or self.proc.returncode is not None:
                return
            try:
                fd = os.open(path, os.O_WRONLY | os.O_NONBLOCK)
            except OSError as e:
                if e.errno != errno.ENXIO:
                    raise
                await asyncio.sleep(0.2)
                continue
            set_pipe_size(fd, MAX_INPUT_BACKLOG)
            self.in_fd = fd
            log(f"[{self.name}] mic pipe connected")
            return
        log(f"[{self.name}] mic pipe never opened - SDL may lack a disk audio driver")

    async def stop(self):
        for t in self._tasks:
            t.cancel()
        self._tasks = []
        if self.proc is not None and self.proc.returncode is None:
            self.proc.terminate()
            try:
                await asyncio.wait_for(self.proc.wait(), timeout=10)
            except asyncio.TimeoutError:
                self.proc.kill()
                await self.proc.wait()
        self.proc = None
        for attr in ("in_fd", "out_fd"):
            fd = getattr(self, attr)
            if fd is not None:
                try:
                    os.close(fd)
                except OSError:
                    pass
                setattr(self, attr, None)
        self.state = "idle"
        self.ready = False

    async def _watch_proc(self):
        rc = await self.proc.wait()
        self.state = "stopped"
        self.ready = False
        await self._emit({"t": "state", "state": "stopped",
                          "msg": f"The engine exited (code {rc}). See the log."})

    # ---------- audio ----------

    def feed(self, i16_bytes: bytes):
        """Mic audio from a browser websocket (int16 @ 24 kHz).

        Chrome has already applied autoGainControl, so this is a light touch: kill
        the low-frequency rumble that laptop mics pick up from the desk, and lift
        a quiet mic a little. Capped at +12 dB so it can't fight the browser's own
        AGC and start pumping.
        """
        pcm = np.frombuffer(i16_to_f32(i16_bytes), dtype="<f4")
        self.feed_f32(self.mic_cond(pcm))

    def feed_f32(self, pcm):
        """Caller audio already as float32 @ 24 kHz - used by the phone path."""
        self.used = True
        self.in_buf.extend(np.asarray(pcm, dtype="<f4").tobytes())
        if len(self.in_buf) > MAX_INPUT_BACKLOG:
            # Model is behind. Drop the oldest audio so latency stays bounded.
            drop = len(self.in_buf) - MAX_INPUT_BACKLOG
            del self.in_buf[:drop]
        self._flush_in()

    def _flush_in(self):
        if self.in_fd is None or not self.in_buf:
            return
        try:
            n = os.write(self.in_fd, self.in_buf)
        except BlockingIOError:
            return
        except OSError:
            self.in_fd = None
            return
        if n:
            del self.in_buf[:n]

    async def _pump_audio(self):
        """Drain the playback pipe at real-time rate. That rate is the backpressure."""
        last = time.monotonic()
        credit = 0.0
        while True:
            await asyncio.sleep(TICK)
            now = time.monotonic()
            credit = min(credit + (now - last) * BYTES_PER_SEC, BYTES_PER_SEC * 0.15)
            last = now
            self._flush_in()

            n = int(credit) & ~3          # whole float32 samples only
            if n <= 0 or self.out_fd is None:
                continue
            try:
                data = os.read(self.out_fd, n)
            except BlockingIOError:
                data = b""
            except OSError:
                data = b""
            if not data:
                continue
            credit -= len(data)
            self.out_bytes += len(data)
            # fps is measured on what the model produced, before any time-stretch
            self._fps_window.append((now, len(data)))
            cutoff = now - 3.0
            while self._fps_window and self._fps_window[0][0] < cutoff:
                self._fps_window.pop(0)

            pcm = self.out_limiter(self.stretch.process(
                np.frombuffer(data, dtype="<f4")))
            if len(pcm):
                await self._emit({"t": "audio", "pcm": f32_to_i16(pcm.tobytes())})

    def set_rate(self, rate: float):
        rate = max(0.5, min(1.5, float(rate)))
        self.rate = rate
        self.stretch.set_rate(rate)

    def context_left(self):
        """Frames of conversation remaining.

        moshi.cpp has no sliding window: it consumes one context frame per 80 ms
        step whether anyone is speaking or not, and when the window fills the
        output degrades into nonsense rather than stopping. So the length of a
        conversation is fixed at start time by -c, and the only honest thing to
        do is show it and end cleanly at the limit.
        """
        if not self.settings:
            return None
        total = int(self.settings[3])
        used = self.out_bytes // FRAME_BYTES_F32
        return max(0, total - used)

    def fps(self) -> float:
        if len(self._fps_window) < 2:
            return 0.0
        span = self._fps_window[-1][0] - self._fps_window[0][0]
        if span <= 0.2:
            return 0.0
        total = sum(b for _, b in self._fps_window)
        return (total / FRAME_BYTES_F32) / span

    # ---------- text ----------

    async def _pump_stdout(self):
        """personaplex prints its transcript token by token, with no newlines."""
        pending = b""
        while True:
            chunk = await self.proc.stdout.read(512)
            if not chunk:
                return
            pending += chunk
            try:
                text = pending.decode("utf-8")
                pending = b""
            except UnicodeDecodeError:
                continue  # split multi-byte char, wait for the rest
            sys.stdout.write(text)
            sys.stdout.flush()
            if not self.ready:
                await self._emit({"t": "log", "v": text})
                if "ready" in text or "done loading" in text:
                    self.ready = True
                    self.state = "ready"
                    took = time.monotonic() - self.started_at
                    await self._emit({"t": "state", "state": "ready",
                                      "msg": f"Engine up in {took:.1f}s. Say hello."})
            else:
                await self._emit({"t": "text", "v": text})

    async def _emit(self, msg):
        cb = self.on_event
        if cb is not None:
            await cb(msg)


async def _null_sink(msg):
    """Where a free engine's output goes."""
    return


class EnginePool:
    """The repo opens one PersonaPlex websocket per BridgeSession, so a two-party
    conference just costs two sockets. Here each session needs its own *process*,
    and every process is a full copy of the weights in VRAM - about 9 GB for q8_0.
    So the pool is capped, and asking for a second engine on a single T4 fails
    loudly instead of OOMing halfway through a call.
    """

    def __init__(self, binary, workdir, model="", rate=1.0, max_engines=1,
                 keep_warm=False):
        self.binary = binary
        self.workdir = workdir
        self.model = model
        self.rate = rate
        self.max_engines = max(1, int(max_engines))
        self.keep_warm = keep_warm
        self.engines = []
        self._lock = asyncio.Lock()
        self._standby_task = None

    def busy(self) -> int:
        return sum(1 for e in self.engines if e.owner is not None)

    def any_busy(self) -> bool:
        return self.busy() > 0

    async def acquire(self, settings, on_event, owner, fresh=True, timeout=300):
        """settings is (prompt, voice, temperature, context, seed).

        A warm engine nobody has spoken to yet counts as fresh - that is what
        makes --keep-warm worth having, because moshi.cpp holds the whole
        conversation in the process and the only way to clear it is a new one.
        """
        async with self._lock:
            for e in self.engines:
                if (e.owner is None and e.state == "ready"
                        and e.settings == settings and (not fresh or not e.used)):
                    log(f"[{e.name}] reusing the warm engine")
                    e.owner = owner
                    e.on_event = on_event
                    e.set_rate(self.rate)
                    return e

            engine = next((e for e in self.engines if e.owner is None), None)
            if engine is None:
                if len(self.engines) >= self.max_engines:
                    log(f"engine pool exhausted ({self.max_engines} in use). "
                        "Raise --max-engines only if the GPU has room for another "
                        "full copy of the weights.")
                    return None
                engine = Engine(self.binary, self.workdir, _null_sink,
                                rate=self.rate, model=self.model,
                                name=f"engine{len(self.engines)}")
                self.engines.append(engine)

            engine.owner = owner
            engine.on_event = on_event
            prompt, voice, temp, ctx, seed = settings
            await engine.start(prompt=prompt, voice=voice, temperature=temp,
                               context=ctx, seed=seed)
            for _ in range(int(timeout * 2)):
                if engine.state == "ready":
                    return engine
                if engine.state == "stopped":
                    engine.owner = None
                    engine.on_event = _null_sink
                    return None
                await asyncio.sleep(0.5)
            engine.owner = None
            engine.on_event = _null_sink
            return None

    async def release(self, engine, restock_settings=None):
        if engine is None:
            return
        engine.owner = None
        engine.on_event = _null_sink
        settings = restock_settings or engine.settings
        await engine.stop()
        if self.keep_warm and settings:
            self._restock(engine, settings)

    def _restock(self, engine, settings):
        """Put a fresh unused engine back on the shelf so the next call starts
        instantly instead of waiting on a cold load."""
        async def _go():
            await asyncio.sleep(1.0)
            if engine.owner is not None:
                return
            log(f"[{engine.name}] restocking a fresh engine for the next call")
            prompt, voice, temp, ctx, seed = settings
            try:
                await engine.start(prompt=prompt, voice=voice, temperature=temp,
                                   context=ctx, seed=seed)
            except Exception as e:
                log("restock failed:", e)
        if self._standby_task and not self._standby_task.done():
            self._standby_task.cancel()
        self._standby_task = asyncio.create_task(_go())

    async def warm(self, settings):
        prompt, voice, temp, ctx, seed = settings
        if not self.engines:
            self.engines.append(Engine(self.binary, self.workdir, _null_sink,
                                       rate=self.rate, model=self.model,
                                       name="engine0"))
        engine = self.engines[0]
        await engine.start(prompt=prompt, voice=voice, temperature=temp,
                           context=ctx, seed=seed)
        for _ in range(600):
            if engine.state == "ready":
                return True
            if engine.state == "stopped":
                return False
            await asyncio.sleep(0.5)
        return False

    async def shutdown(self):
        if self._standby_task:
            self._standby_task.cancel()
        for e in self.engines:
            await e.stop()


# ==========================================================================
# src/bridge-session.ts
#
# One Twilio Media Stream. Twilio speaks 8 kHz mu-law in 20 ms frames; the
# model speaks 24 kHz float32. Everything between those two facts lives here.
# ==========================================================================


class BridgeSession:
    FRAME = TWILIO_FRAME
    METRICS_PERIOD = 2.0

    def __init__(self, twilio_ws, stream_sid, call_sid, pool, role="caller-a",
                 bwe=False, inbound_only=True):
        self.twilio_ws = twilio_ws
        self.stream_sid = stream_sid
        self.call_sid = call_sid
        self.role = role
        self.pool = pool
        self.engine = None
        self.active = False

        # src/bridge-session.ts: output can be redirected at another party's
        # stream, which is how a conference gets each side the other's voice.
        self.output_ws = twilio_ws
        self.output_stream_sid = stream_sid

        self.up = Up8to24()
        self.down = Down24to8()
        self.cond = CallerAudio(max_gain_db=24.0, bwe=bwe)
        self.out_band = HandsetOut()
        self.out_buf = np.zeros(0, dtype=np.float32)

        self.packets_in = 0
        self.packets_out = 0
        self.frames_other = 0          # media frames that were not the caller
        self.inbound_only = inbound_only
        self.last_audio_sent = 0.0
        self.latency_estimate = 0
        self.packet_times = []
        self._metrics_task = None
        self.caller_speaking = False
        self._silence_handle = None
        self._ctx_warned = False

    @property
    def live(self):
        return self.stream_sid is not None and not self.twilio_ws.closed

    def set_output_target(self, ws, stream_sid):
        self.output_ws = ws
        self.output_stream_sid = stream_sid
        log(f"[Bridge] Output for {self.stream_sid} redirected to stream {stream_sid}")

    async def start(self) -> bool:
        settings = call_registry.engine_settings(self.role)
        self.engine = await self.pool.acquire(settings, self._on_engine_event,
                                              owner=self, fresh=True)
        if self.engine is None:
            log(f"[Bridge] No engine available for {self.stream_sid}")
            return False
        # Twilio plays out whatever we send, so a deep buffer becomes delay the
        # caller hears. But too tight and the stretcher has to keep changing rate
        # to stay inside it, which sounds worse than the delay.
        self.engine.stretch.backlog_cap = 1.2
        self.engine.set_rate(float(call_registry.config.get("rate", 1.0)))
        self.active = True
        self._metrics_task = asyncio.create_task(self._metrics_loop())
        log(f"[Bridge] Session active for stream {self.stream_sid}")
        return True

    # ---------- caller -> model ----------

    def handle_twilio_audio(self, mulaw_payload: bytes, track: str = "inbound"):
        if not self.active or self.engine is None:
            return
        # Only the caller's audio may reach the model. An outbound frame is our
        # own voice coming straight back; feeding that in makes her answer
        # herself, and no echo suppressor can help because a digital loopback
        # arrives at exactly the level we sent - it reads as the caller talking
        # loudly.
        if self.inbound_only and track != "inbound":
            self.frames_other += 1
            return

        self.packets_in += 1
        pcm8 = mulaw_decode(mulaw_payload).astype(np.float32) / 32768.0

        # src/bridge-session.ts: RMS gate marks a caller turn in the transcript.
        # The repo compares against 200 on the int16 scale; same threshold here.
        if _rms(pcm8) > (200.0 / 32768.0):
            if not self.caller_speaking:
                self.caller_speaking = True
                call_registry.add_transcript(self.call_sid, "caller", "[speaking]")
            if self._silence_handle:
                self._silence_handle.cancel()
            loop = asyncio.get_event_loop()
            self._silence_handle = loop.call_later(1.5, self._mark_silent)

        # Echo suppression, noise gate, then gain. In that order: a plain AGC
        # amplifies both the line noise and her own voice coming back off the
        # handset up to conversational level, and then she answers herself.
        self.engine.feed_f32(self.cond(self.up(pcm8)))

    def _mark_silent(self):
        self.caller_speaking = False

    # ---------- model -> caller ----------

    async def _on_engine_event(self, msg):
        kind = msg.get("t")
        if kind == "audio":
            await self._send_audio(msg["pcm"])
        elif kind == "text":
            call_registry.add_transcript(self.call_sid, "translation", msg["v"])
        elif kind == "state" and msg.get("state") == "stopped":
            log("[Bridge] engine stopped underneath session", self.stream_sid)
            self.active = False

    async def _send_audio(self, i16_24k_bytes):
        """Model audio (int16 @ 24 kHz) -> 8 kHz mu-law -> Twilio, in 20 ms frames."""
        if not self.active or self.output_ws is None or self.output_ws.closed:
            return

        if self.last_audio_sent > 0:
            self.latency_estimate = int((time.monotonic() - self.last_audio_sent) * 1000)
        now = time.monotonic()
        self.packet_times.append(now)
        if len(self.packet_times) > 50:
            self.packet_times.pop(0)

        pcm = np.frombuffer(i16_24k_bytes, dtype="<i2").astype(np.float32) / 32768.0
        self.out_buf = np.concatenate([self.out_buf, self.down(pcm)])
        while len(self.out_buf) >= self.FRAME:
            chunk, self.out_buf = self.out_buf[:self.FRAME], self.out_buf[self.FRAME:]
            chunk = self.out_band(chunk)
            # Tell the echo suppressor what we just sent, so it knows when the
            # thing arriving from the caller is really us.
            self.cond.note_far(chunk)
            ulaw = mulaw_encode(np.clip(chunk, -1.0, 1.0) * 32767.0)
            try:
                await self.output_ws.send_json({
                    "event": "media",
                    "streamSid": self.output_stream_sid,
                    "media": {"payload": base64.b64encode(ulaw).decode()},
                })
                self.packets_out += 1
                self.last_audio_sent = time.monotonic()
            except (ConnectionResetError, RuntimeError):
                return

    # ---------- metrics ----------

    async def _metrics_loop(self):
        while True:
            await asyncio.sleep(self.METRICS_PERIOD)
            if not self.active:
                return
            if await self._check_context():
                return
            e = self.engine
            ctx_total = int(e.settings[3]) / 12.5 if (e and e.settings) else 0
            call_registry.update_metrics(self.call_sid, {
                "packetsIn": self.packets_in,
                "packetsOut": self.packets_out,
                "latency": self.latency_estimate,
                "jitter": self._jitter(),
                "fps": round(e.fps(), 2) if e else 0,
                "ctxLeft": ((e.context_left() or 0) / 12.5) if e else 0,
                "ctxTotal": ctx_total,
                "echo": bool(self.cond.echo.ducking),
                "gain": round(self.cond.gain, 2),
                "level": round(20 * math.log10(max(self.cond.last_out, 1e-6)), 1),
            })

    async def _check_context(self) -> bool:
        """moshi.cpp has no sliding window: it burns one context frame every
        80 ms whether anyone is talking or not, and past the end it degrades
        into nonsense rather than stopping. So we watch it and end the call
        ourselves. Returns True if the call was ended.
        """
        e = self.engine
        if e is None or not e.settings:
            return False
        left = e.context_left()
        if left is None:
            return False
        secs = left / 12.5

        if secs <= 0:
            log(f"[Bridge] context window full on {self.stream_sid}; ending the call")
            call_registry.add_notice(
                self.call_sid,
                "Ended at the conversation length limit - the context window filled. "
                "Raise Context in Config for a longer call.")
            await self.stop("context-exhausted")
            # With <Connect><Stream>, closing the socket ends the <Connect> verb
            # and, with no TwiML after it, the call itself.
            try:
                await self.twilio_ws.close()
            except Exception:
                pass
            return True

        if secs <= 30 and not self._ctx_warned:
            self._ctx_warned = True
            call_registry.add_notice(
                self.call_sid,
                f"About {secs:.0f}s of conversation left before the context window fills.")
        return False

    def set_bwe(self, on: bool):
        """Toggleable mid-call: the phone band-extension is the one knob worth
        changing while somebody is on the line, because whether it helps depends
        on the handset."""
        self.cond.bwe = BandwidthExtend() if on else None

    def _jitter(self) -> int:
        if len(self.packet_times) < 2:
            return 0
        gaps = np.diff(np.array(self.packet_times)) * 1000.0
        return int(round(float(np.std(gaps))))

    async def stop(self, reason="hangup"):
        if not self.active and self.engine is None:
            return
        self.active = False
        if self._metrics_task:
            self._metrics_task.cancel()
            self._metrics_task = None
        if self._silence_handle:
            self._silence_handle.cancel()
            self._silence_handle = None
        engine, self.engine = self.engine, None
        if engine is not None:
            engine.stretch.backlog_cap = 1.5
            await self.pool.release(engine)
        call_registry.end_call(self.call_sid, reason)
        log(f"[Bridge] Session stopped for stream {self.stream_sid} "
            f"in={self.packets_in} out={self.packets_out} ignored={self.frames_other}")


# ==========================================================================
# src/conference-session.ts
# ==========================================================================

conferences = {}


def get_or_create_conference(name: str) -> dict:
    if name not in conferences:
        conferences[name] = {"callerA": None, "callerB": None,
                             "outboundCallSid": None, "tornDown": False}
    return conferences[name]


async def teardown_conference(name: str, initiator: str):
    conf = conferences.get(name)
    if not conf or conf["tornDown"]:
        return
    conf["tornDown"] = True
    conferences.pop(name, None)
    log(f"[Conference] Tearing down {name} (initiated by {initiator})")

    for key in ("callerA", "callerB"):
        p = conf.get(key)
        if p:
            await p["session"].stop("conference-teardown")

    # Hang up the other party's call through the REST API.
    other = None
    if initiator == "caller-a" and conf.get("callerB"):
        other = conf["callerB"]["callSid"]
    elif initiator == "caller-b" and conf.get("callerA"):
        other = conf["callerA"]["callSid"]
    if other and not twilio_missing():
        try:
            await hangup_call(other)
        except Exception as e:
            log("[Conference] Failed to hang up other leg:", e)

    # Also hang up the outbound call SID if Party B never connected.
    if (initiator == "caller-a" and conf.get("outboundCallSid")
            and not conf.get("callerB") and not twilio_missing()):
        try:
            await hangup_call(conf["outboundCallSid"])
        except Exception as e:
            log("[Conference] Failed to hang up outbound call:", e)


def cross_wire(conf: dict, name: str):
    """A's translated audio goes out on B's stream and vice versa."""
    a, b = conf.get("callerA"), conf.get("callerB")
    if a and b:
        a["session"].set_output_target(b["ws"], b["streamSid"])
        b["session"].set_output_target(a["ws"], a["streamSid"])
        log(f"[Conference] Audio cross-wired for {name}")


# ==========================================================================
# src/outbound-call.ts
# ==========================================================================

def twilio_env():
    """Accepts either this notebook's original secret names or the repo's."""
    e = os.environ.get
    return {
        "accountSid": (e("TWILIO_ACCOUNT_SID") or "").strip(),
        "authToken": (e("TWILIO_AUTH_TOKEN") or "").strip(),
        "apiKeySid": (e("TWILIO_API_KEY_SID") or "").strip(),
        "apiKeySecret": (e("TWILIO_API_KEY_SECRET") or "").strip(),
        "from": (e("TWILIO_PHONE_NUMBER") or e("TWILIO_FROM_NUMBER") or "").strip(),
        "to": (e("TWILIO_TO_NUMBER") or "").strip(),
        "twimlAppSid": (e("TWILIO_TWIML_APP_SID") or "").strip(),
    }


def twilio_client():
    """API key pair if we have one (the repo's way, and what AccessToken needs),
    otherwise the account auth token."""
    from twilio.rest import Client
    cfg = twilio_env()
    if cfg["apiKeySid"] and cfg["apiKeySecret"]:
        return Client(cfg["apiKeySid"], cfg["apiKeySecret"], cfg["accountSid"])
    return Client(cfg["accountSid"], cfg["authToken"])


def twilio_missing(need_webrtc=False):
    cfg = twilio_env()
    missing = []
    if not cfg["accountSid"]:
        missing.append("TWILIO_ACCOUNT_SID")
    if not (cfg["authToken"] or (cfg["apiKeySid"] and cfg["apiKeySecret"])):
        missing.append("TWILIO_AUTH_TOKEN (or TWILIO_API_KEY_SID + TWILIO_API_KEY_SECRET)")
    if not cfg["from"]:
        missing.append("TWILIO_FROM_NUMBER")
    if need_webrtc:
        if not (cfg["apiKeySid"] and cfg["apiKeySecret"]):
            missing.append("TWILIO_API_KEY_SID + TWILIO_API_KEY_SECRET")
        if not cfg["twimlAppSid"]:
            missing.append("TWILIO_TWIML_APP_SID")
    return missing


async def _rest(fn, *a, **kw):
    return await asyncio.to_thread(fn, *a, **kw)


async def create_call(to_number: str, twiml: str) -> str:
    cfg = twilio_env()

    def _dial():
        return twilio_client().calls.create(to=to_number, from_=cfg["from"], twiml=twiml)
    call = await _rest(_dial)
    log(f"[Outbound] Dialing {to_number}, CallSid: {call.sid}")
    return call.sid


async def hangup_call(call_sid: str):
    def _end():
        try:
            twilio_client().calls(call_sid).update(status="completed")
            log(f"[Outbound] Hung up call: {call_sid}")
        except Exception as e:
            if getattr(e, "code", None) == 20404:
                return          # already completed
            raise
    await _rest(_end)


async def dial_party_b(to_number: str, conference_name: str, host: str) -> str:
    """src/outbound-call.ts dialPartyB.

    The repo points Twilio at https://<host>/voice/conference-outbound and lets
    it fetch the TwiML. We hand the same TwiML over inline instead, because a
    free ngrok tunnel shows a browser interstitial on HTML fetches that would
    break the webhook - a websocket upgrade goes straight through. The route
    still exists below for anyone on a real domain.
    """
    twiml = twiml_conference_outbound(host, conference_name)
    return await create_call(to_number, twiml)


# ==========================================================================
# src/server.ts + src/conference-handler.ts - TwiML
# ==========================================================================

def _attr(v) -> str:
    """Always double-quoted. saxutils.quoteattr silently switches to single
    quotes when the value contains a double quote, which is valid XML but makes
    the TwiML harder to eyeball against the repo's."""
    return '"' + xml_escape(str(v), {'"': "&quot;"}) + '"'


def _wss(host: str, path: str) -> str:
    host = host.replace("https://", "").replace("http://", "").rstrip("/")
    return f"wss://{host}{path}"


def twiml_stream(host: str, params: dict = None, say: str = None) -> str:
    """<Connect><Stream> with nested <Parameter> elements.

    Twilio strips query strings off a Stream url, so per-call values have to
    travel as <Parameter> - they arrive in the start event's customParameters.
    """
    lines = []
    for k, v in (params or {}).items():
        if v is None:
            continue
        lines.append(f'      <Parameter name={_attr(k)} value={_attr(v)} />')
    inner = "\n".join(lines)
    url = _attr(_wss(host, "/media-stream"))
    stream = (f'    <Stream url={url}>\n{inner}\n    </Stream>' if inner
              else f'    <Stream url={url} />')
    say_el = f"  <Say>{xml_escape(say)}</Say>\n" if say else ""
    return ('<?xml version="1.0" encoding="UTF-8"?>\n'
            f"<Response>\n{say_el}  <Connect>\n{stream}\n  </Connect>\n</Response>")


def twiml_inbound(host: str) -> str:
    return twiml_stream(host, say="Connecting you now.")


def twiml_webrtc(host: str, mode: str = "single", role: str = "caller-a") -> str:
    if mode == "conference":
        return twiml_stream(host, {"role": role,
                                   "conference": f"conf-{now_ms()}"})
    return twiml_stream(host)


def twiml_conference_inbound(host: str, conference_name: str = None) -> str:
    conference_name = conference_name or f"translate-{now_ms()}"
    return twiml_stream(host, {"role": "caller-a", "conference": conference_name},
                        say="Connecting you to the conference. "
                            "Dialing the other party now.")


def twiml_conference_outbound(host: str, conference_name: str) -> str:
    return twiml_stream(host, {"role": "caller-b", "conference": conference_name},
                        say="You are being connected to a translated call.")


# ==========================================================================
# src/index.ts - the server, the media-stream socket and the monitor socket
# ==========================================================================


class BrowserSession:
    METRICS_PERIOD = 2.0

    """Not in the repo. The repo's browser call is a WebRTC leg that goes out to
    Twilio and comes back as a Media Stream, which means the audio is 8 kHz
    mu-law by the time the model hears it. That path is supported below
    (/voice/webrtc), but this one keeps a browser talking to the engine at the
    model's own 24 kHz, which is strictly better when nobody needs a phone. It
    registers in the same CallRegistry so the dashboard shows both the same way.
    """

    def __init__(self, ws, pool, call_sid):
        self.ws = ws
        self.pool = pool
        self.call_sid = call_sid
        self.engine = None
        self.active = False
        self._metrics_task = None
        self._ctx_warned = False

    async def start(self, settings) -> bool:
        self.engine = await self.pool.acquire(settings, self._on_engine_event,
                                              owner=self, fresh=True)
        if self.engine is None:
            return False
        self.engine.set_rate(float(call_registry.config.get("rate", 1.0)))
        self.active = True
        self._metrics_task = asyncio.create_task(self._metrics_loop())
        return True

    def feed(self, i16_bytes):
        if self.active and self.engine:
            self.engine.feed(i16_bytes)

    async def _on_engine_event(self, msg):
        if self.ws is None or self.ws.closed:
            return
        try:
            if msg["t"] == "audio":
                await self.ws.send_bytes(msg["pcm"])
            else:
                if msg["t"] == "text":
                    call_registry.add_transcript(self.call_sid, "translation", msg["v"])
                await self.ws.send_json(msg)
        except (ConnectionResetError, RuntimeError):
            pass

    async def _metrics_loop(self):
        while self.active:
            await asyncio.sleep(self.METRICS_PERIOD)
            e = self.engine
            if not e:
                return
            if await self._check_context():
                return
            call_registry.update_metrics(self.call_sid, {
                "fps": round(e.fps(), 2),
                "ctxLeft": (e.context_left() or 0) / 12.5,
                "ctxTotal": int(e.settings[3]) / 12.5 if e.settings else 0,
                "gain": round(e.mic_cond.gain, 2),
            })

    async def _check_context(self) -> bool:
        e = self.engine
        left = e.context_left() if (e and e.settings) else None
        if left is None:
            return False
        secs = left / 12.5
        if secs <= 0:
            msg = ("Conversation reached its length limit. Start a new one - "
                   "raise Context in Config for longer.")
            call_registry.add_notice(self.call_sid, msg)
            await self.stop("context-exhausted")
            if self.ws is not None and not self.ws.closed:
                try:
                    await self.ws.send_json({"t": "state", "state": "stopped", "msg": msg})
                except (ConnectionResetError, RuntimeError):
                    pass
            return True
        if secs <= 30 and not self._ctx_warned:
            self._ctx_warned = True
            call_registry.add_notice(
                self.call_sid,
                f"About {secs:.0f}s of conversation left before the context window fills.")
        return False

    async def stop(self, reason="hangup"):
        self.active = False
        if self._metrics_task:
            self._metrics_task.cancel()
            self._metrics_task = None
        engine, self.engine = self.engine, None
        if engine is not None:
            await self.pool.release(engine)
        call_registry.end_call(self.call_sid, reason)


class Bridge:
    def __init__(self, binary, workdir, ui_file, rate=1.0, public_url=None,
                 keep_warm=False, bwe=False, model="", max_engines=1):
        self.ui_file = ui_file
        self.pool = EnginePool(binary, workdir, model=model, rate=rate,
                               max_engines=max_engines, keep_warm=keep_warm)
        self.bwe = bwe
        self.public_url = public_url
        self.seen_host = None
        self.monitor_clients = set()
        self.sessions = set()          # live BridgeSessions, for live config changes
        self.browser_ws = None
        self.loop = None
        call_registry.on_monitor(self._broadcast)

    # ---------- monitor fan-out (src/monitor-ws.ts) ----------

    def _broadcast(self, event: dict):
        if not self.monitor_clients:
            return
        msg = json.dumps(event)
        loop = self.loop or asyncio.get_event_loop()
        for ws in list(self.monitor_clients):
            if ws.closed:
                self.monitor_clients.discard(ws)
                continue
            loop.create_task(self._safe_send(ws, msg))

    async def _safe_send(self, ws, msg):
        try:
            await ws.send_str(msg)
        except (ConnectionResetError, RuntimeError):
            self.monitor_clients.discard(ws)

    # ---------- hosts ----------

    def note_host(self, request):
        self.seen_host = (request.headers.get("X-Forwarded-Host")
                          or request.headers.get("Host") or request.host)

    def base_host(self):
        if self.public_url:
            return self.public_url.replace("https://", "").replace("http://", "").rstrip("/")
        return self.seen_host

    def base_url(self):
        host = self.base_host()
        return f"https://{host}" if host else None

    # ---------- /media-stream (src/index.ts wss.on('connection')) ----------

    async def media_stream_ws(self, request):
        self.note_host(request)
        ws = web.WebSocketResponse(heartbeat=20, max_msg_size=4 * 1024 * 1024)
        await ws.prepare(request)
        log("[MediaStream] Client connected")

        session = None
        conference_name = None
        role = None
        is_conference = False

        try:
            async for msg in ws:
                if msg.type != WSMsgType.TEXT:
                    continue
                data = json.loads(msg.data)
                event = data.get("event")

                if event == "connected":
                    log("[MediaStream] Connected event received")

                elif event == "start":
                    start = data["start"]
                    stream_sid = start["streamSid"]
                    call_sid = start.get("callSid") or stream_sid
                    params = start.get("customParameters") or {}
                    tracks = start.get("tracks") or ["inbound"]
                    log(f"[Server] Stream started: {stream_sid} (call: {call_sid}), "
                        f"tracks: {','.join(tracks)}, customParameters: {params}")

                    role = params.get("role")
                    conference_name = params.get("conference")
                    is_conference = bool(role and conference_name)

                    if is_conference:
                        log(f"[Server] Conference mode: {role} in {conference_name}")
                        call_registry.register_call(call_sid, stream_sid,
                                                    "conference", role, conference_name)
                        session = BridgeSession(ws, stream_sid, call_sid, self.pool,
                                                role=role, bwe=self.bwe)
                        conf = get_or_create_conference(conference_name)
                        entry = {"ws": ws, "session": session,
                                 "streamSid": stream_sid, "callSid": call_sid}
                        if role == "caller-a":
                            conf["callerA"] = entry
                            log(f"[Conference] Caller A joined: {conference_name}")
                        else:
                            conf["callerB"] = entry
                            log(f"[Conference] Caller B joined: {conference_name}")
                    else:
                        log("[Server] Single-call mode")
                        call_registry.register_call(call_sid, stream_sid,
                                                    "single", None, None)
                        session = BridgeSession(ws, stream_sid, call_sid, self.pool,
                                                role="caller-a", bwe=self.bwe)

                    ok = await session.start()
                    if ok:
                        self.sessions.add(session)
                    if not ok:
                        log("[Server] Failed to start the engine for this call")
                        call_registry.end_call(call_sid, "engine-unavailable")
                        await session.stop("engine-unavailable")
                        session = None
                        await ws.close()
                        break

                    if is_conference:
                        conf = get_or_create_conference(conference_name)
                        if role == "caller-a" and not conf.get("outboundCallSid"):
                            phone = call_registry.config.get("partyBPhone", "")
                            host = self.base_host()
                            if phone and host:
                                try:
                                    sid = await dial_party_b(phone, conference_name, host)
                                    conf["outboundCallSid"] = sid
                                    log(f"[Conference] Dialing party B at {phone}")
                                except Exception as e:
                                    log("[Conference] Failed to dial party B:", e)
                            else:
                                log("[Conference] No party B phone configured")
                        cross_wire(conf, conference_name)

                elif event == "media" and session is not None:
                    media = data["media"]
                    session.handle_twilio_audio(base64.b64decode(media["payload"]),
                                                media.get("track", "inbound"))

                elif event == "stop":
                    log("[MediaStream] Stream stopped")
                    break
        except Exception as e:
            log("[MediaStream] error:", e)
        finally:
            if session is not None:
                self.sessions.discard(session)
                if session.packets_out == 0:
                    log("WARNING: no audio was sent to the caller.")
                await session.stop()
            if is_conference and conference_name:
                await teardown_conference(conference_name, role or "unknown")
            log("[MediaStream] Client disconnected")
        return ws

    # ---------- /ws/monitor (src/monitor-ws.ts) ----------

    async def monitor_ws(self, request):
        self.note_host(request)
        ws = web.WebSocketResponse(heartbeat=25)
        await ws.prepare(request)
        self.monitor_clients.add(ws)
        await ws.send_json({"type": "snapshot", "calls": call_registry.get_all_calls()})
        await ws.send_json({"type": "config:updated", "config": call_registry.get_config()})
        try:
            async for msg in ws:
                if msg.type != WSMsgType.TEXT:
                    continue
                try:
                    await self._monitor_command(ws, json.loads(msg.data))
                except Exception as e:
                    log("[Monitor] Invalid message:", e)
        finally:
            self.monitor_clients.discard(ws)
        return ws

    async def _monitor_command(self, ws, msg):
        kind = msg.get("type")

        if kind == "config:get":
            await ws.send_json({"type": "config:updated",
                                "config": call_registry.get_config()})

        elif kind == "config:set":
            cfg = call_registry.update_config(msg.get("config") or {})
            # Two settings take effect without a restart. Everything else -
            # prompt, voice, temperature, seed, context - is baked into the
            # process at launch, so it applies to the next call.
            for e in self.pool.engines:
                e.set_rate(float(cfg.get("rate", 1.0)))
            self.bwe = bool(cfg.get("bwe"))
            for sess in list(self.sessions):
                sess.set_bwe(self.bwe)

        elif kind == "token:request":
            # src/monitor-ws.ts token:request - a Voice SDK access token
            cfg = twilio_env()
            missing = twilio_missing(need_webrtc=True)
            if missing:
                await ws.send_json({"type": "token:error",
                                    "reason": "Missing: " + ", ".join(missing)})
                return
            try:
                from twilio.jwt.access_token import AccessToken
                from twilio.jwt.access_token.grants import VoiceGrant
                token = AccessToken(cfg["accountSid"], cfg["apiKeySid"],
                                    cfg["apiKeySecret"],
                                    identity=msg.get("identity", "browser"))
                token.add_grant(VoiceGrant(outgoing_application_sid=cfg["twimlAppSid"],
                                           incoming_allow=True))
                jwt = token.to_jwt()
                await ws.send_json({"type": "token:response",
                                    "token": jwt if isinstance(jwt, str) else jwt.decode()})
            except Exception as e:
                await ws.send_json({"type": "token:error", "reason": str(e)})

        elif kind == "call:dial":
            # Not in the repo: dial a phone from the dashboard, the way the old
            # "Call me" button did. Inline TwiML, same shape the webhook returns.
            await self._dial_out(ws, msg)

        elif kind == "call:hangup":
            sid = msg.get("callSid")
            if sid:
                try:
                    await hangup_call(sid)
                except Exception as e:
                    log("hangup failed:", e)

    async def _dial_out(self, ws, msg):
        missing = twilio_missing()
        if missing:
            await ws.send_json({"type": "dial:error",
                                "reason": "Missing secrets: " + ", ".join(missing)})
            return
        host = self.base_host()
        if not host:
            await ws.send_json({"type": "dial:error",
                                "reason": "No public URL yet - open this page "
                                          "through the ngrok link, not localhost."})
            return
        cfg = twilio_env()
        to = (msg.get("to") or cfg["to"] or "").strip()
        if not to:
            await ws.send_json({"type": "dial:error",
                                "reason": "No number to dial. Set TWILIO_TO_NUMBER."})
            return

        mode = call_registry.config.get("mode", "single")
        if mode == "conference":
            name = f"translate-{now_ms()}"
            twiml = twiml_conference_inbound(host, name)
        else:
            twiml = twiml_stream(host)

        try:
            sid = await create_call(to, twiml)
            await ws.send_json({"type": "dial:ok", "callSid": sid, "to": to})
        except Exception as e:
            code = getattr(e, "code", None)
            hint = {
                21219: "That number isn't verified on your trial account.",
                21210: "The From number isn't one of your Twilio numbers.",
                21212: "The From number is invalid or not voice-capable.",
                20003: "Authentication failed - check the SID and auth token.",
                21205: "The From number can't make calls on this account.",
            }.get(code, "")
            await ws.send_json({"type": "dial:error",
                                "reason": f"Twilio error {code or '?'}: {e} {hint}".strip()})

    # ---------- /ws - direct browser audio ----------

    async def browser_audio_ws(self, request):
        self.note_host(request)
        ws = web.WebSocketResponse(max_msg_size=8 * 1024 * 1024, heartbeat=20)
        await ws.prepare(request)
        if self.browser_ws is not None and not self.browser_ws.closed:
            await ws.send_json({"t": "state", "state": "busy",
                                "msg": "Another tab already has the line. Close it and reload."})
            await ws.close()
            return ws

        self.browser_ws = ws
        session = None
        log("client connected from", request.remote, "host", self.seen_host)
        await ws.send_json({"t": "state", "state": "idle", "msg": "Ready when you are."})
        try:
            async for msg in ws:
                if msg.type == WSMsgType.BINARY:
                    if session:
                        session.feed(msg.data)
                elif msg.type == WSMsgType.TEXT:
                    cmd = json.loads(msg.data)
                    kind = cmd.get("t")
                    if kind == "start":
                        if session:
                            await session.stop("restart")
                        call_sid = f"browser-{now_ms()}"
                        call_registry.register_call(call_sid, call_sid, "single",
                                                    "browser", None)
                        session = BrowserSession(ws, self.pool, call_sid)
                        settings = call_registry.engine_settings("caller-a")
                        await ws.send_json({"t": "state", "state": "starting",
                                            "msg": "Loading the model."})
                        ok = await session.start(settings)
                        await ws.send_json(
                            {"t": "state", "state": "ready", "msg": "Say hello."} if ok else
                            {"t": "state", "state": "stopped",
                             "msg": "No engine available. Is a phone call using it?"})
                        if not ok:
                            await session.stop("engine-unavailable")
                            session = None
                    elif kind == "stop":
                        if session:
                            await session.stop()
                            session = None
                        await ws.send_json({"t": "state", "state": "idle",
                                            "msg": "Call ended. Next one starts fresh."})
                    elif kind == "rate":
                        call_registry.update_config({"rate": float(cmd.get("v", 1.0))})
                        for e in self.pool.engines:
                            e.set_rate(float(cmd.get("v", 1.0)))
                    elif kind == "bwe":
                        self.bwe = bool(cmd.get("v"))
                        call_registry.update_config({"bwe": self.bwe})
                    elif kind == "ping":
                        await ws.send_json({"t": "pong", "id": cmd.get("id")})
                elif msg.type == WSMsgType.ERROR:
                    break
        finally:
            if session:
                await session.stop()
            self.browser_ws = None
        return ws

    # ---------- http (src/server.ts, src/api-routes.ts) ----------

    async def index(self, request):
        self.note_host(request)
        return web.FileResponse(self.ui_file)

    async def api_calls(self, request):
        return web.json_response({"calls": call_registry.get_all_calls()})

    async def api_config_get(self, request):
        return web.json_response(call_registry.get_config())

    async def api_config_put(self, request):
        body = await request.json()
        return web.json_response(call_registry.update_config(body))

    async def api_meta(self, request):
        """Voices, sample rate, and what the phone features need."""
        self.note_host(request)
        cfg = twilio_env()
        return web.json_response({
            "voices": VOICES,
            "sampleRate": SAMPLE_RATE,
            "twilioNumber": cfg["from"],
            "toNumber": cfg["to"][-4:] if cfg["to"] else "",
            "dialReady": not twilio_missing(),
            "dialReason": ", ".join(twilio_missing()),
            "webrtcReady": not twilio_missing(need_webrtc=True),
            "webrtcReason": ", ".join(twilio_missing(need_webrtc=True)),
            "publicUrl": self.base_url() or "",
            "maxEngines": self.pool.max_engines,
            "streamUrl": _wss(self.base_host() or "", "/media-stream")
                         if self.base_host() else "",
        })

    async def voice_inbound(self, request):
        self.note_host(request)
        return web.Response(text=twiml_inbound(self.base_host() or request.host),
                            content_type="text/xml")

    async def voice_webrtc(self, request):
        self.note_host(request)
        body = await request.post()
        mode = body.get("mode") or body.get("Mode") or "single"
        role = body.get("role") or body.get("Role") or "caller-a"
        log("[TwiML /voice/webrtc] mode:", mode, "role:", role)
        return web.Response(text=twiml_webrtc(self.base_host() or request.host,
                                              mode, role),
                            content_type="text/xml")

    async def voice_conference(self, request):
        self.note_host(request)
        return web.Response(
            text=twiml_conference_inbound(self.base_host() or request.host),
            content_type="text/xml")

    async def voice_conference_outbound(self, request):
        self.note_host(request)
        name = request.query.get("conference") or f"translate-{now_ms()}"
        return web.Response(
            text=twiml_conference_outbound(self.base_host() or request.host, name),
            content_type="text/xml")

    # ---------- housekeeping ----------

    async def _janitor(self, app):
        async def _go():
            while True:
                await asyncio.sleep(60)
                call_registry.prune()
        task = asyncio.create_task(_go())
        yield
        task.cancel()
        await self.pool.shutdown()


async def selftest(binary, workdir, voice, prompt, seconds, rate=1.0,
                   model="", context=3000):
    """Run the whole pipe rig without a browser: feed silence, see if audio comes back."""
    got = {"bytes": 0, "text": ""}

    async def sink(msg):
        if msg["t"] == "audio":
            got["bytes"] += len(msg["pcm"])
        elif msg["t"] == "text":
            got["text"] += msg["v"]

    eng = Engine(binary, workdir, sink, rate=rate, model=model, name="selftest")
    t0 = time.monotonic()
    await eng.start(prompt, voice, 0.8, context)

    # Wait for the load before starting the clock. q8_0 is 8.9 GB and the first
    # read off disk can take minutes, which would otherwise swallow the whole
    # test budget and report a failure that isn't one.
    load_deadline = time.monotonic() + 600
    while time.monotonic() < load_deadline:
        if eng.state == "ready":
            print(f"engine ready in {time.monotonic() - t0:.0f}s")
            break
        if eng.state == "stopped":
            print("FAIL - the engine exited before it was ready; see the output above.")
            await eng.stop()
            return
        await asyncio.sleep(0.5)
    else:
        print("FAIL - the engine never reported ready.")
        await eng.stop()
        return

    deadline = time.monotonic() + seconds
    silence = (np.zeros(FRAME_SAMPLES, dtype="<i2")).tobytes()
    next_frame = time.monotonic()
    while time.monotonic() < deadline:
        if eng.proc is None or eng.proc.returncode is not None:
            print("\nengine exited early")
            break
        if eng.in_fd is not None:
            eng.feed(silence)
        next_frame += 0.08
        await asyncio.sleep(max(0.0, next_frame - time.monotonic()))

    secs = got["bytes"] / (SAMPLE_RATE * 2)
    print("\n" + "=" * 62)
    print(f"audio returned : {got['bytes']} bytes  ({secs:.1f}s of speech)")
    print(f"frame rate     : {eng.fps():.2f} fps   (12.5 = real time)")
    print(f"pace           : {rate:.2f}x")
    print(f"transcript     : {got['text'][:400] or '(nothing)'}")
    print("=" * 62)
    if got["bytes"] == 0:
        print("FAIL - nothing came back through the playback pipe.")
        print("       SDL probably has no disk audio driver. See the notebook's")
        print("       troubleshooting section.")
    else:
        print("PASS - the pipe rig works. Start the server.")
    await eng.stop()


def build_app(bridge: "Bridge") -> web.Application:
    app = web.Application(client_max_size=8 * 1024 * 1024)
    app.add_routes([
        web.get("/", bridge.index),

        # src/api-routes.ts
        web.get("/api/calls", bridge.api_calls),
        web.get("/api/config", bridge.api_config_get),
        web.put("/api/config", bridge.api_config_put),
        web.get("/api/meta", bridge.api_meta),

        # src/server.ts + src/conference-handler.ts
        web.route("*", "/voice/inbound", bridge.voice_inbound),
        web.route("*", "/voice/webrtc", bridge.voice_webrtc),
        web.route("*", "/voice/conference", bridge.voice_conference),
        web.route("*", "/voice/conference-outbound", bridge.voice_conference_outbound),

        # src/index.ts upgrade paths
        web.get("/media-stream", bridge.media_stream_ws),
        web.post("/media-stream", bridge.media_stream_ws),
        web.get("/ws/monitor", bridge.monitor_ws),

        # local high-fidelity browser path
        web.get("/ws", bridge.browser_audio_ws),
    ])
    app.cleanup_ctx.append(bridge._janitor)
    return app


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--binary", required=True)
    ap.add_argument("--workdir", required=True)
    ap.add_argument("--model", default="",
                    help="model directory relative to --workdir; empty means the "
                         "binary's own default (the q4_k folder)")
    ap.add_argument("--ui", default=str(Path(__file__).with_name("index.html")))
    ap.add_argument("--host", default="0.0.0.0")
    ap.add_argument("--port", type=int, default=8998)
    ap.add_argument("--selftest", type=float, default=0,
                    help="seconds; run without a browser and report")
    ap.add_argument("--voice", default="NATF1")
    ap.add_argument("--prompt", default="")
    ap.add_argument("--bwe", action="store_true",
                    help="synthesise energy above 3.4 kHz on caller audio")
    ap.add_argument("--keep-warm", action="store_true",
                    help="load the model at startup and leave it resident")
    ap.add_argument("--warm-prompt", default="",
                    help="prompt to warm up with; the config panel overrides it per call")
    ap.add_argument("--warm-voice", default="NATF1")
    ap.add_argument("--warm-context", type=int, default=3000,
                    help="context frames; 12.5 per second of conversation, max 3000")
    ap.add_argument("--public-url", default=None,
                    help="override the https URL Twilio should dial back into")
    ap.add_argument("--rate", type=float, default=1.0,
                    help="speaking pace; 0.75 = three quarter speed, pitch unchanged")
    ap.add_argument("--max-engines", type=int, default=1,
                    help="concurrent personaplex processes. Conference mode needs 2, "
                         "and each one is a full copy of the weights in VRAM.")
    ap.add_argument("--party-b", default="",
                    help="phone number for the second leg of a conference call")
    args = ap.parse_args()

    if args.selftest:
        asyncio.run(selftest(args.binary, args.workdir, args.voice,
                             args.prompt, args.selftest, args.rate,
                             model=args.model, context=args.warm_context))
        return

    # seed the registry config from the command line
    seed_cfg = {"context": args.warm_context, "rate": args.rate, "bwe": args.bwe}
    if args.warm_prompt:
        seed_cfg["textPrompt"] = args.warm_prompt
    if args.warm_voice:
        seed_cfg["voicePrompt"] = args.warm_voice
    if args.party_b:
        seed_cfg["partyBPhone"] = args.party_b
    if args.max_engines >= 2 and os.environ.get("CALL_MODE"):
        seed_cfg["mode"] = os.environ["CALL_MODE"]
    call_registry.config.update(seed_cfg)

    bridge = Bridge(args.binary, args.workdir, args.ui, rate=args.rate,
                    public_url=args.public_url, keep_warm=args.keep_warm,
                    bwe=args.bwe, model=args.model, max_engines=args.max_engines)
    app = build_app(bridge)

    async def _remember_loop(_app):
        bridge.loop = asyncio.get_running_loop()
    app.on_startup.append(_remember_loop)

    if args.keep_warm:
        async def _warm(_app):
            log("warming the model up")
            ok = await bridge.pool.warm(call_registry.engine_settings("caller-a"))
            log("model warm" if ok else "warm-up failed; it will retry on first use")
        app.on_startup.append(_warm)

    log(f"serving on http://{args.host}:{args.port}")
    log(f"  media stream : wss://<public-host>/media-stream")
    log(f"  voice webhook: https://<public-host>/voice/inbound")
    log(f"  engines      : up to {args.max_engines}")
    web.run_app(app, host=args.host, port=args.port, print=None)


if __name__ == "__main__":
    main()

Writing /kaggle/working/agent/bridge.py


## Cell 7 - The web UI

`index.html`. The repo's React client - TopBar, Sidebar, CallDetail, MetricsGrid,
Transcript, Dialer, ConfigPanel - as one page with no build step, because a notebook
can't run Vite. Same layout, same event reducer, same auto-saving config panel.

The Twilio Voice SDK comes from jsDelivr: Twilio stopped publishing the 2.x SDK to
its own CDN, so the npm build is the only way to get it into a plain page.

In [7]:
%%writefile /kaggle/working/agent/index.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Twilio &rarr; NVIDIA PersonaPlex</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Barlow+Semi+Condensed:wght@400;500;600;700&family=IBM+Plex+Mono:wght@400;500;600&display=swap" rel="stylesheet">
<style>
:root{
  --bg:#0E1216; --panel:#161C22; --panel2:#1B232A; --rule:#232C34;
  --fg:#DCE4EB; --muted:#7A8895; --dim:#4C5964;
  --you:#F2A63B; --her:#4EC9D4; --red:#E5484D; --ok:#5BC98B; --accent:#6E7BF2;
  --disp:"Barlow Semi Condensed",-apple-system,"Segoe UI",sans-serif;
  --mono:"IBM Plex Mono",ui-monospace,"SF Mono",Menlo,monospace;
}
*{box-sizing:border-box}
html,body{height:100%}
body{margin:0;background:var(--bg);color:var(--fg);font-family:var(--disp);
  font-size:16px;line-height:1.45;-webkit-font-smoothing:antialiased;
  display:flex;flex-direction:column;overflow:hidden}
button,select,textarea,input{font-family:inherit;color:inherit}
:focus-visible{outline:2px solid var(--her);outline-offset:2px}

/* ---------- TopBar ---------- */
header.top{display:flex;align-items:center;justify-content:space-between;gap:14px;
  padding:10px 16px;background:var(--panel);border-bottom:1px solid var(--rule);flex:none}
.brand{display:flex;align-items:center;gap:10px;background:none;border:0;cursor:pointer;padding:0}
.brand h1{font-size:19px;font-weight:600;letter-spacing:.02em;margin:0;white-space:nowrap}
.brand:hover{opacity:.85}
.dot{width:9px;height:9px;border-radius:50%;background:var(--dim);flex:none}
.dot.live{background:var(--ok)} .dot.warn{background:var(--you)} .dot.bad{background:var(--red)}
.sub{font-family:var(--mono);font-size:10.5px;color:var(--muted);letter-spacing:.05em}
.nav{display:flex;gap:6px}
.nav button{border:1px solid var(--rule);background:var(--panel2);color:var(--muted);
  padding:6px 14px;border-radius:3px;font-size:14px;cursor:pointer}
.nav button:hover{color:var(--fg)}
.nav button.on{background:var(--accent);border-color:var(--accent);color:#fff}

/* ---------- layout ---------- */
.body{flex:1;display:flex;min-height:0}
aside{width:290px;flex:none;background:var(--panel);border-right:1px solid var(--rule);
  overflow-y:auto;padding:12px}
main{flex:1;overflow-y:auto;padding:20px}
@media(max-width:860px){
  aside{display:none} .body.showlist aside{display:block;width:100%}
  .body.showlist main{display:none}
}
.eyebrow{font-family:var(--mono);font-size:10px;letter-spacing:.16em;
  text-transform:uppercase;color:var(--muted)}

/* ---------- Sidebar ---------- */
.sect{margin-bottom:16px}
.sect h3{margin:0 0 8px}
.callcard{display:block;width:100%;text-align:left;padding:8px 9px;margin-bottom:5px;
  border-radius:3px;border:1px solid var(--rule);background:var(--panel2);cursor:pointer}
.callcard:hover{border-color:var(--muted)}
.callcard.sel{border-color:var(--accent)}
.callcard.act{border-color:#2C5F45}
.callcard.dim{opacity:.55}
.callcard .sid{font-family:var(--mono);font-size:11px;color:#C3CDD6;
  overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
.callcard .meta{font-family:var(--mono);font-size:9.5px;color:var(--muted);margin-top:2px}
.empty{color:var(--dim);font-size:12.5px;font-family:var(--mono)}

/* ---------- panels ---------- */
.panel{background:var(--panel);border:1px solid var(--rule);border-radius:4px}
.panel+.panel{margin-top:14px}
.phead{display:flex;justify-content:space-between;align-items:center;gap:10px;
  padding:9px 14px;border-bottom:1px solid var(--rule)}
.pbody{padding:14px}
.hero{max-width:860px;margin:0 auto}

/* ---------- CallDetail ---------- */
.detailhead{display:flex;align-items:flex-start;justify-content:space-between;
  gap:14px;margin-bottom:14px}
.detailhead h2{margin:0;font-family:var(--mono);font-size:16px;font-weight:600}
.detailhead p{margin:3px 0 0;font-family:var(--mono);font-size:11px;color:var(--muted)}
.badge{padding:3px 10px;border-radius:999px;font-size:11.5px;font-weight:600;
  font-family:var(--mono);letter-spacing:.08em}
.badge.live{background:#12331F;color:#7FE0A7} .badge.end{background:var(--panel2);color:var(--muted)}
.metrics{display:grid;grid-template-columns:repeat(6,1fr);gap:8px;margin-bottom:14px}
@media(max-width:760px){.metrics{grid-template-columns:repeat(3,1fr)}}
.metric{background:var(--panel);border:1px solid var(--rule);border-radius:4px;
  padding:9px 6px;text-align:center}
.metric span{display:block;font-family:var(--mono);font-size:9px;letter-spacing:.12em;
  text-transform:uppercase;color:var(--muted)}
.metric b{display:block;font-family:var(--mono);font-size:16px;font-weight:600;
  margin-top:3px;font-variant-numeric:tabular-nums}
.metric b.warn{color:var(--you)} .metric b.bad{color:var(--red)} .metric b.ok{color:var(--ok)}
.transcript{padding:12px 14px;height:min(46vh,420px);overflow-y:auto}
.line{display:flex;gap:10px;margin-bottom:11px}
.line .pip{width:7px;height:7px;border-radius:50%;flex:none;margin-top:7px}
.line.caller .pip{background:var(--you)} .line.agent .pip{background:var(--her)}
.line .who{font-family:var(--mono);font-size:9.5px;letter-spacing:.12em;
  text-transform:uppercase}
.line.caller .who{color:var(--you)} .line.agent .who{color:var(--her)}
.line p{margin:2px 0 0;font-size:14.5px;line-height:1.6;white-space:pre-wrap;
  word-break:break-word}
.line .ts{font-family:var(--mono);font-size:9.5px;color:var(--dim);flex:none;margin-top:6px}
.ital{color:var(--dim);font-style:italic}

/* ---------- levels ---------- */
.lvl{display:flex;align-items:center;gap:10px;padding:6px 14px}
.lvlname{font-family:var(--mono);font-size:10px;letter-spacing:.1em;text-transform:uppercase;
  width:48px;flex:none;color:var(--muted)}
.lvlbar{flex:1;height:6px;background:#0A0E11;border:1px solid var(--rule);border-radius:2px;overflow:hidden}
.lvlfill{height:100%;width:0}
.lvlfill.you{background:var(--you)} .lvlfill.her{background:var(--her)}
.lvlnum{font-family:var(--mono);font-size:10px;color:var(--dim);width:32px;text-align:right;
  flex:none;font-variant-numeric:tabular-nums}

/* ---------- forms ---------- */
.field{margin-bottom:14px}
label{display:block;font-family:var(--mono);font-size:10px;letter-spacing:.13em;
  text-transform:uppercase;color:var(--muted);margin-bottom:6px}
textarea,select,input[type=number],input[type=text],input[type=tel]{width:100%;
  background:#0A0E11;color:var(--fg);border:1px solid var(--rule);border-radius:3px;
  padding:9px 10px;font-size:14px}
textarea{font-family:var(--mono);font-size:12.5px;line-height:1.6;resize:vertical;min-height:140px}
textarea:focus,select:focus,input:focus{border-color:var(--accent);outline:none}
input[type=range]{width:100%;accent-color:var(--her)}
.row{display:flex;gap:10px}.row>*{flex:1;min-width:0}
.hint{font-family:var(--mono);font-size:10.5px;color:var(--dim);margin-top:5px;line-height:1.55}
.btn{border:1px solid var(--rule);background:var(--panel2);color:var(--fg);
  padding:10px 18px;border-radius:3px;font-size:15px;font-weight:600;letter-spacing:.02em;cursor:pointer}
.btn:hover:not(:disabled){border-color:var(--muted)}
.btn:disabled{opacity:.4;cursor:not-allowed}
.btn.go{background:var(--ok);border-color:var(--ok);color:#06210F}
.btn.end{background:transparent;border-color:var(--red);color:var(--red)}
.btn.sm{padding:6px 12px;font-size:13px;font-weight:500}
.btn.full{width:100%;display:flex;align-items:center;justify-content:center;gap:9px}
.toggle{display:flex;gap:8px;flex-wrap:wrap}
.toggle button{border:1px solid var(--rule);background:var(--panel2);color:var(--muted);
  padding:8px 14px;border-radius:3px;font-size:14px;cursor:pointer}
.toggle button.on{background:var(--accent);border-color:var(--accent);color:#fff}
.notice{background:#2B2210;border:1px solid #5E4A1E;border-radius:3px;
  padding:9px 12px;color:#E8C878;font-size:13px;margin-bottom:12px;font-family:var(--mono)}
.err{background:#2A1114;border:1px solid #5E2126;border-radius:3px;padding:10px 12px;
  color:#F1868A;font-size:13.5px;margin-top:12px}
.ok{color:var(--ok);font-family:var(--mono);font-size:11px}
.info{background:var(--panel2);border-radius:3px;padding:11px 12px;font-size:13.5px;
  color:#B7C2CC;line-height:1.6}
.info b{color:var(--fg)}
.num{font-family:var(--mono);font-size:19px;color:var(--fg)}
details summary{cursor:pointer;padding:9px 14px;font-family:var(--mono);font-size:10px;
  letter-spacing:.14em;text-transform:uppercase;color:var(--muted)}
details[open] summary{border-bottom:1px solid var(--rule)}
pre.log{margin:0;padding:12px 14px;max-height:170px;overflow:auto;font-family:var(--mono);
  font-size:11px;line-height:1.6;color:var(--muted);white-space:pre-wrap;word-break:break-all}
.spin{width:15px;height:15px;border:2px solid rgba(255,255,255,.3);border-top-color:#fff;
  border-radius:50%;animation:sp .7s linear infinite}
@keyframes sp{to{transform:rotate(360deg)}}
.hide{display:none !important}
</style>
</head>
<body>

<!-- ============ TopBar (client/src/components/TopBar.tsx) ============ -->
<header class="top">
  <button class="brand" id="brand">
    <h1>Twilio &rarr; NVIDIA PersonaPlex</h1>
    <span class="dot" id="connDot"></span>
    <span class="sub" id="connText">connecting</span>
  </button>
  <nav class="nav">
    <button data-view="calls" class="on">Calls</button>
    <button data-view="dialer">Call</button>
    <button data-view="config">Config</button>
  </nav>
</header>

<div class="body" id="body">

  <!-- ============ Sidebar (client/src/components/Sidebar.tsx) ============ -->
  <aside>
    <div class="sect">
      <h3 class="eyebrow">Active calls</h3>
      <div id="activeList"><p class="empty">No active calls</p></div>
    </div>
    <div class="sect">
      <h3 class="eyebrow">Recent</h3>
      <div id="recentList"><p class="empty">No recent calls</p></div>
    </div>
  </aside>

  <main>

    <!-- ============ CallDetail (client/src/components/CallDetail.tsx) ============ -->
    <section id="view-calls">
      <div id="callEmpty" class="hero" style="padding-top:8vh;text-align:center">
        <h2 style="margin:0 0 8px;font-size:24px;font-weight:600">Real-time speech to speech</h2>
        <p style="color:var(--muted);font-size:14.5px;max-width:440px;margin:0 auto 26px">
          Caller audio is streamed through PersonaPlex and streamed straight back.
          Start a call to see the transcript and the media metrics.
        </p>
        <div style="max-width:360px;margin:0 auto;display:flex;flex-direction:column;gap:14px">
          <div class="panel" style="padding:14px" id="numberBox">
            <div class="eyebrow" style="margin-bottom:4px">Call the Twilio number</div>
            <div class="num" id="twilioNumber">&mdash;</div>
          </div>
          <div class="eyebrow">or</div>
          <button class="btn go" id="goDial">Start a call</button>
        </div>
      </div>

      <div id="callDetail" class="hero hide">
        <div class="detailhead">
          <div>
            <h2 id="dSid">&mdash;</h2>
            <p id="dMeta">&mdash;</p>
          </div>
          <div style="display:flex;align-items:center;gap:10px">
            <span class="badge end" id="dBadge">ENDED</span>
            <button class="btn sm end hide" id="dHangup">Hang up</button>
          </div>
        </div>

        <div class="notice hide" id="dNotice"></div>

        <!-- MetricsGrid (client/src/components/MetricsGrid.tsx) -->
        <div class="metrics">
          <div class="metric"><span>Uptime</span><b id="mUptime">0:00</b></div>
          <div class="metric"><span>Latency</span><b id="mLat">0ms</b></div>
          <div class="metric"><span>Jitter</span><b id="mJit">0ms</b></div>
          <div class="metric"><span>Packets</span><b id="mPkt">0</b></div>
          <div class="metric"><span>Frame rate</span><b id="mFps">0.0</b></div>
          <div class="metric"><span>Ctx left</span><b id="mCtx">&mdash;</b></div>
        </div>

        <!-- Transcript (client/src/components/Transcript.tsx) -->
        <div class="panel">
          <div class="phead">
            <span class="eyebrow">Live transcript</span>
            <div style="display:flex;gap:7px">
              <button class="btn sm" id="copyBtn">Copy</button>
              <button class="btn sm" id="saveBtn">Save</button>
            </div>
          </div>
          <div class="transcript" id="transcript"><p class="empty">Waiting for speech&hellip;</p></div>
          <div class="lvl" id="lvlRow1">
            <div class="lvlname">You</div>
            <div class="lvlbar"><div class="lvlfill you" id="lvlYou"></div></div>
            <div class="lvlnum" id="numYou">&minus;&infin;</div>
          </div>
          <div class="lvl" style="padding-bottom:11px">
            <div class="lvlname">Agent</div>
            <div class="lvlbar"><div class="lvlfill her" id="lvlHer"></div></div>
            <div class="lvlnum" id="numHer">&minus;&infin;</div>
          </div>
        </div>

        <div class="panel">
          <details><summary>Engine log</summary><pre class="log" id="log">waiting for the bridge</pre></details>
        </div>
      </div>
    </section>

    <!-- ============ Dialer (client/src/components/Dialer.tsx) ============ -->
    <section id="view-dialer" class="hide">
      <div class="hero" style="max-width:560px">
        <h2 style="margin:0 0 18px;font-size:21px;font-weight:600">Start a call</h2>

        <div class="panel"><div class="pbody">
          <div class="field">
            <label>How</label>
            <div class="toggle" id="howToggle">
              <button data-how="browser" class="on">Browser (direct)</button>
              <button data-how="webrtc">Browser via Twilio</button>
              <button data-how="phone">Ring my phone</button>
            </div>
            <div class="hint" id="howHint"></div>
          </div>

          <div class="field">
            <label>Mode</label>
            <div class="toggle" id="modeToggle">
              <button data-mode="single" class="on">Single call</button>
              <button data-mode="conference">Conference</button>
            </div>
            <div class="hint" id="modeHint"></div>
          </div>

          <div class="field hide" id="roleField">
            <label>Join as</label>
            <div class="toggle" id="roleToggle">
              <button data-role="caller-a" class="on">Caller A</button>
              <button data-role="caller-b">Caller B</button>
            </div>
          </div>

          <div class="field hide" id="confBox">
            <div class="info" id="confInfo"></div>
          </div>

          <button class="btn go full" id="dialBtn">Connect</button>
          <button class="btn end full hide" id="hangBtn" style="margin-top:10px">Hang up</button>
          <div class="err hide" id="dialErr"></div>
          <div class="hint" style="margin-top:12px">
            Wear headphones for a browser call &mdash; the model is full duplex and will
            answer its own voice coming out of your speakers.
          </div>
        </div></div>
      </div>
    </section>

    <!-- ============ ConfigPanel (client/src/components/ConfigPanel.tsx) ============ -->
    <section id="view-config" class="hide">
      <div class="hero" style="max-width:640px">
        <div style="display:flex;align-items:center;justify-content:space-between;margin-bottom:16px">
          <h2 style="margin:0;font-size:21px;font-weight:600">Call configuration</h2>
          <span class="ok hide" id="savedTag">Saved</span>
        </div>

        <div class="panel"><div class="pbody">
          <div class="field">
            <label>Mode</label>
            <div class="toggle" id="cfgMode">
              <button data-mode="single" class="on">Single call</button>
              <button data-mode="conference">Conference</button>
            </div>
          </div>
          <div class="field">
            <label for="preset">Preset</label>
            <select id="preset"></select>
          </div>
          <div class="field">
            <label for="textPrompt">System prompt</label>
            <textarea id="textPrompt" spellcheck="false"></textarea>
            <div class="hint">Tuned on "You work for {company} which is a {industry} and your
              name is {name}. Information: &hellip;". Facts after <b>Information:</b> are the only
              things she can say &mdash; anything missing gets invented.</div>
          </div>
          <div class="row field">
            <div>
              <label for="voicePrompt">Voice</label>
              <select id="voicePrompt"></select>
            </div>
            <div>
              <label for="targetLanguage">Target language</label>
              <select id="targetLanguage"></select>
            </div>
          </div>
          <div class="hint" style="margin:-8px 0 14px">Picking a language rewrites the prompt
            with the interpreter template from the repo. Pick a persona preset afterwards to
            go back to a sales agent.</div>
          <div class="row field">
            <div>
              <label for="context">Context (frames)</label>
              <input type="number" id="context" min="400" max="3000" step="100" value="3000">
            </div>
            <div>
              <label for="seed">Seed (&minus;1 = random)</label>
              <input type="number" id="seed" min="-1" step="1" value="-1">
            </div>
          </div>
          <div class="hint" id="ctxHint" style="margin:-8px 0 14px"></div>
          <div class="field">
            <label for="temperature">Temperature <span id="tempVal" style="color:var(--fg)">0.80</span></label>
            <input type="range" id="temperature" min="0.4" max="1.0" step="0.05" value="0.8">
            <div class="hint">Lower sticks to the script. Higher improvises.</div>
          </div>
          <div class="field">
            <label for="rate">Speaking pace <span id="rateVal" style="color:var(--fg)">1.00x</span></label>
            <input type="range" id="rate" min="0.7" max="1.0" step="0.05" value="1.0">
            <div class="hint">1.00 is untouched audio. Below that she is time-stretched after
              generation &mdash; pitch preserved. Takes effect mid-call.</div>
          </div>
          <div class="field" style="margin-bottom:0">
            <label><input type="checkbox" id="bwe" style="width:auto;margin-right:7px">Band extension (phone only)</label>
            <div class="hint">Phone lines stop at 3.4 kHz; the model was trained on full-band
              speech. This synthesises the missing top end on caller audio.</div>
          </div>
        </div></div>

        <div class="panel hide" id="partyBPanel">
          <div class="phead"><span class="eyebrow">Conference &mdash; party B</span></div>
          <div class="pbody">
            <div class="field">
              <label for="partyBPhone">Party B phone</label>
              <input type="tel" id="partyBPhone" placeholder="+919876543210">
              <div class="hint">E.164 only. On a trial account it must be a verified number.</div>
            </div>
            <div class="row field">
              <div>
                <label for="voicePromptB">Party B voice</label>
                <select id="voicePromptB"></select>
              </div>
              <div>
                <label for="targetLanguageB">B &rarr; A language</label>
                <select id="targetLanguageB"></select>
              </div>
            </div>
            <div class="field" style="margin-bottom:0">
              <label for="textPromptB">B &rarr; A prompt</label>
              <textarea id="textPromptB" style="min-height:90px" spellcheck="false"></textarea>
            </div>
          </div>
        </div>

        <div class="panel" id="engineWarn"><div class="pbody">
          <div class="eyebrow" style="margin-bottom:7px">Engine capacity</div>
          <div class="info" id="engineInfo">&mdash;</div>
        </div></div>
      </div>
    </section>

  </main>
</div>

<script src="https://cdn.jsdelivr.net/npm/@twilio/voice-sdk@2.18.3/dist/twilio.min.js"></script>
<script>
"use strict";
const SR = 24000, MIC_CHUNK = 480;
const $ = id => document.getElementById(id);
const BASE = location.pathname.replace(/[^\/]*$/, "");
const esc = s => String(s).replace(/[&<>"]/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;"}[c]));
const fmtDur = s => { const m = Math.floor(s/60), x = Math.floor(s%60);
  return m + ":" + String(x).padStart(2,"0"); };

const LANGUAGES = ["English","Spanish","French","German","Japanese","Mandarin",
                   "Portuguese","Italian","Korean","Arabic","Hindi"];

const PRESETS = {
  "Riya - Piyush Academy":
    "You work for Piyush Academy which is an online education institute and your name is Riya. " +
    "Information: Basic Computer Course is four thousand rupees over two months. " +
    "Tally with GST is seven thousand rupees over three months. " +
    "Full Stack Web Development is twenty five thousand rupees over six months. " +
    "Classes are live online and recordings are provided. " +
    "A free demo class is available before you enroll. There is twenty percent off this month. " +
    "Ask what they want to learn and how much time they can give each day, then recommend one " +
    "course and offer to book the free demo class. Keep your answers to one or two short " +
    "sentences and let them speak. If you do not know something, say you will have the team confirm it.",
  "Maria - TransLingo interpreter":
    "You work for TransLingo which is a live interpretation service and your name is Maria. " +
    "You are a Spanish interpreter. Your ONLY job is to repeat exactly what the caller says, " +
    "but in Spanish. Do NOT answer questions. Do NOT have a conversation. Do NOT add anything. " +
    "Just translate their exact words into Spanish, nothing else.",
  "Elena - Aurora Fitness":
    "You work for Aurora Fitness which is a home gym equipment company and your name is Elena Marsh. " +
    "Information: The Starter Bundle with adjustable dumbbells and a mat is two hundred and forty nine dollars. " +
    "The Home Studio with dumbbells, a bench and a resistance kit is five hundred and forty nine dollars. " +
    "The Pro Setup with a full rack, barbell and bench is eleven hundred and ninety nine dollars. " +
    "Shipping is free over five hundred dollars and returns are accepted for thirty days. " +
    "Ask about their goals and how much space they have, then recommend one package. " +
    "Keep your answers to one or two short sentences and let them speak.",
  "Blank template":
    "You work for COMPANY which is a INDUSTRY and your name is NAME. " +
    "Information: FACT ONE. FACT TWO. FACT THREE. " +
    "Ask QUESTION, then recommend one option. " +
    "Keep your answers to one or two short sentences and let them speak."
};

/* ==================================================================
   State - a port of client/src/hooks/useMonitorSocket.ts
   ================================================================== */
const S = {
  calls: [], config: null, connected: false, meta: null,
  view: "calls", selected: null,
  how: "browser", mode: "single", role: "caller-a",
  browserCall: false, dialing: false, phoneSid: null,
};
let mon = null, monTimer = null;

function applyEvent(ev){
  switch(ev.type){
    case "snapshot": S.calls = ev.calls || []; break;
    case "call:start":
      if(!S.calls.find(c => c.callSid === ev.callSid)){
        S.calls.push({callSid:ev.callSid, streamSid:ev.streamSid, mode:ev.mode,
          role:ev.role, conferenceName:null, status:"active", startTime:ev.startTime,
          endTime:null, duration:0, transcript:[], notice:"",
          metrics:{uptime:0,packetsIn:0,packetsOut:0,latency:0,jitter:0,fps:0,ctxLeft:0,ctxTotal:0}});
      }
      S.selected = ev.callSid; S.view = "calls"; setView("calls");
      break;
    case "call:text": {
      const c = S.calls.find(c => c.callSid === ev.callSid); if(!c) break;
      const last = c.transcript[c.transcript.length-1];
      if(last && last.speaker === ev.speaker) last.text = ev.text;
      else c.transcript.push({speaker:ev.speaker, text:ev.text, timestamp:ev.timestamp});
      break;
    }
    case "call:notice": {
      const c = S.calls.find(c => c.callSid === ev.callSid); if(!c) break;
      c.notice = ev.text;
      break;
    }
    case "call:metrics": {
      const c = S.calls.find(c => c.callSid === ev.callSid); if(!c) break;
      c.duration = ev.uptime; c.metrics = Object.assign({}, c.metrics, ev);
      break;
    }
    case "call:end": {
      const c = S.calls.find(c => c.callSid === ev.callSid); if(!c) break;
      c.status = "ended"; c.duration = ev.duration; c.endTime = Date.now();
      break;
    }
    case "config:updated": S.config = ev.config; fillConfig(ev.config); break;
    case "token:error": setDialError(ev.reason); break;
    case "dial:error": S.dialing = false; setDialError(ev.reason); renderDialer(); break;
    case "dial:ok":
      S.dialing = false; S.phoneSid = ev.callSid;
      setDialError(""); renderDialer();
      break;
  }
  render();
}

function connectMonitor(){
  const proto = location.protocol === "https:" ? "wss" : "ws";
  mon = new WebSocket(`${proto}://${location.host}${BASE}ws/monitor`);
  mon.onopen = () => { S.connected = true; render();
                       mon.send(JSON.stringify({type:"config:get"})); };
  mon.onclose = () => { S.connected = false; render();
                        clearTimeout(monTimer); monTimer = setTimeout(connectMonitor, 2500); };
  mon.onmessage = e => { try { applyEvent(JSON.parse(e.data)); } catch(err){} };
}
const send = m => { if(mon && mon.readyState === 1) mon.send(JSON.stringify(m)); };

/* ==================================================================
   Render
   ================================================================== */
function render(){
  $("connDot").className = "dot" + (S.connected ? " live" : " bad");
  $("connText").textContent = S.connected ? "connected" : "reconnecting";

  const active = S.calls.filter(c => c.status === "active");
  const recent = S.calls.filter(c => c.status === "ended").slice(-12).reverse();
  $("activeList").innerHTML = active.length ? active.map(c => card(c,false)).join("")
    : '<p class="empty">No active calls</p>';
  $("recentList").innerHTML = recent.length ? recent.map(c => card(c,true)).join("")
    : '<p class="empty">No recent calls</p>';

  const call = S.calls.find(c => c.callSid === S.selected) || null;
  $("callEmpty").classList.toggle("hide", !!call);
  $("callDetail").classList.toggle("hide", !call);
  if(call) renderDetail(call);
}

function card(c, dim){
  const cls = "callcard" + (c.callSid === S.selected ? " sel" : "")
            + (c.status === "active" ? " act" : "") + (dim ? " dim" : "");
  return `<button class="${cls}" data-sid="${esc(c.callSid)}">
    <div class="sid">${esc(c.callSid)}</div>
    <div class="meta">${esc(c.mode)}${c.role ? " · " + esc(c.role) : ""} · ${fmtDur(c.duration||0)}</div>
  </button>`;
}

function renderDetail(c){
  $("dSid").textContent = c.callSid;
  $("dMeta").textContent = `${c.mode}${c.role ? " · " + c.role : ""} · started `
    + new Date(c.startTime).toLocaleTimeString();
  $("dBadge").textContent = c.status === "active" ? "LIVE" : "ENDED";
  $("dBadge").className = "badge " + (c.status === "active" ? "live" : "end");
  $("dHangup").classList.toggle("hide", c.status !== "active");

  $("dNotice").textContent = c.notice || "";
  $("dNotice").classList.toggle("hide", !c.notice);

  const m = c.metrics || {};
  $("mUptime").textContent = fmtDur(m.uptime || 0);
  $("mLat").textContent = Math.round(m.latency || 0) + "ms";
  $("mLat").className = m.latency > 400 ? "bad" : m.latency > 200 ? "warn" : "";
  $("mJit").textContent = Math.round(m.jitter || 0) + "ms";
  $("mPkt").textContent = (m.packetsIn || 0) + (m.packetsOut || 0);
  const f = m.fps || 0;
  $("mFps").textContent = f.toFixed(1);
  $("mFps").className = f === 0 ? "" : f < 12.2 ? "bad" : "ok";
  $("mCtx").textContent = m.ctxTotal ? fmtDur(Math.max(0, m.ctxLeft || 0)) : "—";
  $("mCtx").className = (m.ctxTotal && m.ctxLeft < 30) ? "bad"
                      : (m.ctxTotal && m.ctxLeft < 60) ? "warn" : "";

  const el = $("transcript");
  if(!c.transcript.length){ el.innerHTML = '<p class="empty">Waiting for speech…</p>'; return; }
  const near = el.scrollHeight - el.scrollTop - el.clientHeight < 60;
  el.innerHTML = c.transcript.map(e => {
    const caller = e.speaker === "caller";
    const body = e.text === "[speaking]"
      ? '<span class="ital">Audio input detected (no ASR available)</span>'
      : (esc(e.text) || '<span class="ital">…</span>');
    return `<div class="line ${caller ? "caller" : "agent"}">
      <div class="pip"></div>
      <div style="min-width:0;flex:1">
        <span class="who">${caller ? "Caller" : "PersonaPlex"}</span>
        <p>${body}</p>
      </div>
      <span class="ts">${new Date(e.timestamp).toLocaleTimeString([], {hour:"2-digit",minute:"2-digit",second:"2-digit"})}</span>
    </div>`;
  }).join("");
  if(near) el.scrollTop = el.scrollHeight;
}

function setView(v){
  S.view = v;
  for(const s of ["calls","dialer","config"]) $("view-"+s).classList.toggle("hide", s !== v);
  document.querySelectorAll(".nav button").forEach(b =>
    b.classList.toggle("on", b.dataset.view === v));
}
document.querySelectorAll(".nav button").forEach(b =>
  b.addEventListener("click", () => setView(b.dataset.view)));
$("brand").addEventListener("click", () => { S.selected = null; setView("calls"); render(); });
$("goDial").addEventListener("click", () => setView("dialer"));
document.addEventListener("click", e => {
  const c = e.target.closest(".callcard");
  if(c){ S.selected = c.dataset.sid; setView("calls"); render(); }
});

$("copyBtn").addEventListener("click", () => {
  navigator.clipboard.writeText($("transcript").innerText).then(() => {
    $("copyBtn").textContent = "Copied"; setTimeout(() => $("copyBtn").textContent = "Copy", 1200);
  });
});
$("saveBtn").addEventListener("click", () => {
  const a = document.createElement("a");
  a.href = URL.createObjectURL(new Blob([$("transcript").innerText], {type:"text/plain"}));
  a.download = "transcript-" + new Date().toISOString().slice(0,19).replace(/[:T]/g,"-") + ".txt";
  a.click(); URL.revokeObjectURL(a.href);
});

/* ==================================================================
   ConfigPanel - client/src/components/ConfigPanel.tsx
   Rendered once, then only populated: rebuilding it on every event
   would steal focus mid-keystroke.
   ================================================================== */
let cfgLoaded = false, saveTimer = null, applying = false;

function fillSelect(el, values, selected){
  el.innerHTML = values.map(v => `<option${v === selected ? " selected" : ""}>${esc(v)}</option>`).join("");
}

function fillConfig(c){
  applying = true;
  if(!cfgLoaded){
    fillSelect($("preset"), ["—"].concat(Object.keys(PRESETS)), "—");
    fillSelect($("targetLanguage"), LANGUAGES, c.targetLanguage);
    fillSelect($("targetLanguageB"), LANGUAGES, c.targetLanguageB);
    cfgLoaded = true;
  }
  $("textPrompt").value = c.textPrompt || "";
  $("textPromptB").value = c.textPromptB || "";
  $("partyBPhone").value = c.partyBPhone || "";
  $("context").value = c.context;
  $("seed").value = c.seed;
  $("temperature").value = c.temperature; $("tempVal").textContent = (+c.temperature).toFixed(2);
  $("rate").value = c.rate; $("rateVal").textContent = (+c.rate).toFixed(2) + "x";
  $("bwe").checked = !!c.bwe;
  $("targetLanguage").value = c.targetLanguage;
  $("targetLanguageB").value = c.targetLanguageB;
  document.querySelectorAll("#cfgMode button").forEach(b =>
    b.classList.toggle("on", b.dataset.mode === c.mode));
  $("partyBPanel").classList.toggle("hide", c.mode !== "conference");
  updateCtxHint();
  if(S.meta){
    fillSelect($("voicePrompt"), S.meta.voices, (c.voicePrompt||"").replace(/\.pt$/,""));
    fillSelect($("voicePromptB"), S.meta.voices, (c.voicePromptB||"").replace(/\.pt$/,""));
  }
  applying = false;
}

function updateCtxHint(){
  const s = (+$("context").value) / 12.5;
  $("ctxHint").textContent = `${$("context").value} frames = ${fmtDur(s)} of conversation. `
    + "One frame goes every 80 ms whether anyone is talking or not, and the conversation "
    + "ends cleanly when the window fills.";
}

function collectConfig(){
  return {
    mode: document.querySelector("#cfgMode button.on").dataset.mode,
    textPrompt: $("textPrompt").value,
    textPromptB: $("textPromptB").value,
    voicePrompt: $("voicePrompt").value,
    voicePromptB: $("voicePromptB").value,
    targetLanguage: $("targetLanguage").value,
    targetLanguageB: $("targetLanguageB").value,
    partyBPhone: $("partyBPhone").value.trim(),
    context: parseInt($("context").value, 10),
    seed: parseInt($("seed").value, 10),
    temperature: parseFloat($("temperature").value),
    rate: parseFloat($("rate").value),
    bwe: $("bwe").checked,
  };
}

function saveConfig(extra){
  if(applying) return;
  const cfg = Object.assign(collectConfig(), extra || {});
  S.config = Object.assign({}, S.config, cfg);
  clearTimeout(saveTimer);
  saveTimer = setTimeout(() => {
    send({type:"config:set", config: cfg});
    $("savedTag").classList.remove("hide");
    setTimeout(() => $("savedTag").classList.add("hide"), 1400);
  }, 300);
  renderDialer();
}

["textPrompt","textPromptB","partyBPhone","voicePrompt","voicePromptB",
 "context","seed","bwe"].forEach(id => $(id).addEventListener("input", () => {
   if(id === "context") updateCtxHint();
   saveConfig();
 }));
$("temperature").addEventListener("input", e => {
  $("tempVal").textContent = (+e.target.value).toFixed(2); saveConfig();
});
$("rate").addEventListener("input", e => {
  $("rateVal").textContent = (+e.target.value).toFixed(2) + "x"; saveConfig();
});
// Picking a language rewrites the prompt from the template, exactly as
// ConfigPanel.tsx does. The server does the same thing on its side.
$("targetLanguage").addEventListener("change", () => { $("textPrompt").value = ""; saveConfig({textPrompt: ""}); });
$("targetLanguageB").addEventListener("change", () => saveConfig());
$("preset").addEventListener("change", e => {
  const p = PRESETS[e.target.value];
  if(p){ $("textPrompt").value = p; saveConfig({textPrompt: p}); }
});
document.querySelectorAll("#cfgMode button").forEach(b => b.addEventListener("click", () => {
  document.querySelectorAll("#cfgMode button").forEach(x => x.classList.remove("on"));
  b.classList.add("on");
  $("partyBPanel").classList.toggle("hide", b.dataset.mode !== "conference");
  S.mode = b.dataset.mode;
  renderDialer();
  saveConfig();
}));

/* ==================================================================
   Dialer - client/src/components/Dialer.tsx + useTwilioDevice.ts
   ================================================================== */
function setDialError(msg){
  const el = $("dialErr");
  el.classList.toggle("hide", !msg);
  el.textContent = msg || "";
}

function renderDialer(){
  const c = S.config || {};
  const m = S.meta || {};
  document.querySelectorAll("#howToggle button").forEach(b =>
    b.classList.toggle("on", b.dataset.how === S.how));
  document.querySelectorAll("#modeToggle button").forEach(b =>
    b.classList.toggle("on", b.dataset.mode === S.mode));
  document.querySelectorAll("#roleToggle button").forEach(b =>
    b.classList.toggle("on", b.dataset.role === S.role));

  $("howHint").textContent =
    S.how === "browser" ? "Your mic goes straight to the model at 24 kHz. Highest quality, "
      + "no Twilio minutes, no extra credentials."
    : S.how === "webrtc" ? "A WebRTC leg out to Twilio and back in as a Media Stream - the "
      + "same 8 kHz path a real phone takes. Needs an API key pair and a TwiML App."
    : "Twilio rings your number and connects the call to this bridge.";

  $("modeHint").textContent = S.mode === "single"
    ? "One caller, one model."
    : "Two legs, each with its own model, cross-wired so each side hears the other translated.";

  $("roleField").classList.toggle("hide", S.mode !== "conference");
  $("confBox").classList.toggle("hide", S.mode !== "conference");
  if(S.mode === "conference"){
    const phone = c.partyBPhone || "";
    $("confInfo").innerHTML = phone
      ? `Caller A speaks <b>${esc(c.targetLanguageB||"English")}</b> → heard as
         <b>${esc(c.targetLanguage||"Spanish")}</b> by caller B.<br>
         Party B will be dialled at <b>${esc(phone)}</b>.`
      : `<span style="color:var(--you)">Set a party B phone number in Config first.</span>`;
  }

  const engines = m.maxEngines || 1;
  const needTwo = S.mode === "conference" && engines < 2;
  const blocked =
    needTwo ? `Conference needs two engines; this server was started with --max-engines ${engines}.`
    : (S.mode === "conference" && !c.partyBPhone) ? "Set a party B phone number in Config first."
    : (S.how === "webrtc" && m.webrtcReady === false) ? "Missing: " + m.webrtcReason
    : (S.how === "phone" && m.dialReady === false) ? "Missing: " + m.dialReason
    : "";

  const btn = $("dialBtn");
  const busy = S.dialing || S.browserCall;
  btn.disabled = !!blocked || busy;
  btn.innerHTML = busy
    ? '<span class="spin"></span>' + (S.browserCall ? "On call" : "Connecting…")
    : S.how === "phone" ? "Ring my phone" : "Connect";
  $("hangBtn").classList.toggle("hide", !S.browserCall);
  if(blocked) setDialError(blocked); else if(!S.dialing) setDialError("");

  $("engineInfo").innerHTML = `This server runs up to <b>${engines}</b> personaplex
    process${engines === 1 ? "" : "es"}. Each one is a full copy of the weights in VRAM,
    so a conference (two engines) needs a card with room for both.`;
}

document.querySelectorAll("#howToggle button").forEach(b =>
  b.addEventListener("click", () => { S.how = b.dataset.how; renderDialer(); }));
document.querySelectorAll("#modeToggle button").forEach(b =>
  b.addEventListener("click", () => {
    S.mode = b.dataset.mode;
    document.querySelectorAll("#cfgMode button").forEach(x =>
      x.classList.toggle("on", x.dataset.mode === S.mode));
    $("partyBPanel").classList.toggle("hide", S.mode !== "conference");
    saveConfig(); renderDialer();
  }));
document.querySelectorAll("#roleToggle button").forEach(b =>
  b.addEventListener("click", () => { S.role = b.dataset.role; renderDialer(); }));

$("dialBtn").addEventListener("click", async () => {
  setDialError("");
  if(S.how === "phone"){
    S.dialing = true; renderDialer();
    send({type:"call:dial"});
    setTimeout(() => { if(S.dialing){ S.dialing = false; renderDialer(); } }, 20000);
  } else if(S.how === "webrtc"){
    S.dialing = true; renderDialer();
    try { await twilioConnect(S.mode, S.role); }
    catch(e){ setDialError(e.message || String(e)); }
    S.dialing = false; renderDialer();
  } else {
    await startBrowserCall();
  }
});
$("hangBtn").addEventListener("click", () => endBrowserCall(false));
$("dHangup").addEventListener("click", () => {
  const c = S.calls.find(x => x.callSid === S.selected);
  if(!c) return;
  if(String(c.callSid).startsWith("browser-")) endBrowserCall(false);
  else if(twCall) twCall.disconnect();
  else send({type:"call:hangup", callSid: c.callSid});
});

/* ---- Twilio Voice SDK leg (useTwilioDevice.ts) ---- */
let twDevice = null, twCall = null;

function requestToken(){
  return new Promise((resolve, reject) => {
    const timeout = setTimeout(() => {
      mon.removeEventListener("message", handler);
      reject(new Error("Token request timed out. Check the server connection."));
    }, 10000);
    const handler = e => {
      let d; try { d = JSON.parse(e.data); } catch { return; }
      if(d.type === "token:response"){
        clearTimeout(timeout); mon.removeEventListener("message", handler); resolve(d.token);
      } else if(d.type === "token:error"){
        clearTimeout(timeout); mon.removeEventListener("message", handler);
        reject(new Error(d.reason));
      }
    };
    mon.addEventListener("message", handler);
    send({type:"token:request", identity: "browser-" + Date.now()});
  });
}

async function twilioConnect(mode, role){
  if(typeof Twilio === "undefined" || !Twilio.Device)
    throw new Error("The Twilio Voice SDK did not load. Check the browser's network tab.");
  const token = await requestToken();
  twDevice = new Twilio.Device(token, {codecPreferences:["opus","pcmu"]});
  await twDevice.register();
  twCall = await twDevice.connect({params:{mode, role}});
  twCall.on("disconnect", () => { twCall = null;
    if(twDevice){ twDevice.destroy(); twDevice = null; } renderDialer(); });
  twCall.on("error", err => { setDialError(err && err.message || "Call error."); });
}

/* ==================================================================
   Direct browser audio - the notebook's own high-fidelity path (/ws)
   ================================================================== */
const CAP_SRC = `
class Cap extends AudioWorkletProcessor{
  constructor(){super();this.n=0;this.buf=new Float32Array(${MIC_CHUNK});}
  process(inp){
    const ch=inp[0] && inp[0][0]; if(!ch) return true;
    for(let i=0;i<ch.length;i++){
      this.buf[this.n++]=ch[i];
      if(this.n===this.buf.length){this.port.postMessage(this.buf.slice());this.n=0;}
    }
    return true;
  }
}
registerProcessor('cap',Cap);`;

const PLAY_SRC = `
class Play extends AudioWorkletProcessor{
  constructor(){
    super();
    this.q=[]; this.queued=0; this.cur=null; this.pos=0;
    this.preroll=Math.round(sampleRate*0.12);
    this.priming=true;
    this.gain=0; this.rampIn=1/Math.round(sampleRate*0.01);
    this.hist=new Float32Array(1024); this.hp=0;
    this.underruns=0; this.tick=0;
    this.port.onmessage=e=>{
      const d=e.data;
      if(d==='flush'){this.q=[];this.queued=0;this.cur=null;this.pos=0;
                      this.priming=true;this.gain=0;}
      else {this.q.push(d); this.queued+=d.length;}
    };
  }
  conceal(out,i,n){
    const len=n-i;
    for(let k=0;k<len;k++){
      out[i+k]=this.hist[this.hp]*(1-(k+1)/len);
      this.hp=(this.hp+1)%this.hist.length;
    }
  }
  process(_,outputs){
    const out=outputs[0][0], n=out.length;
    if(this.priming){
      if(this.queued<this.preroll){out.fill(0);this.report();return true;}
      this.priming=false; this.gain=0;
    }
    let i=0;
    while(i<n){
      if(!this.cur){this.cur=this.q.shift()||null;this.pos=0;}
      if(!this.cur){this.underruns++;this.conceal(out,i,n);this.priming=true;break;}
      const take=Math.min(n-i,this.cur.length-this.pos);
      for(let k=0;k<take;k++){
        if(this.gain<1) this.gain=Math.min(1,this.gain+this.rampIn);
        const v=this.cur[this.pos+k]*this.gain;
        out[i+k]=v;
        this.hist[this.hp]=v; this.hp=(this.hp+1)%this.hist.length;
      }
      i+=take; this.pos+=take; this.queued-=take;
      if(this.pos>=this.cur.length) this.cur=null;
    }
    this.report();
    return true;
  }
  report(){
    if((this.tick++%8)===0)
      this.port.postMessage({queued:this.queued,underruns:this.underruns});
  }
}
registerProcessor('play',Play);`;

const blobURL = src => URL.createObjectURL(new Blob([src], {type:"application/javascript"}));
function resample(x, from, to){
  if(from === to) return x;
  const ratio = from/to, n = Math.round(x.length/ratio), y = new Float32Array(n);
  for(let i=0;i<n;i++){
    const p=i*ratio, i0=p|0, f=p-i0;
    y[i]=x[i0]*(1-f)+(x[Math.min(i0+1,x.length-1)]||0)*f;
  }
  return y;
}
const toI16 = f => { const a=new Int16Array(f.length);
  for(let i=0;i<f.length;i++){const v=Math.max(-1,Math.min(1,f[i]));a[i]=v*32767;} return a; };
const toF32 = a => { const f=new Float32Array(a.length);
  for(let i=0;i<a.length;i++) f[i]=a[i]/32768; return f; };
const rmsDb = f => { let s=0; for(let i=0;i<f.length;i++) s+=f[i]*f[i];
  const r=Math.sqrt(s/f.length); return r<1e-5 ? -Infinity : 20*Math.log10(r); };

function setLevel(which, f){
  const db = rmsDb(f);
  const pct = db === -Infinity ? 0 : Math.max(0, Math.min(100, (db+55)/55*100));
  $(which === "you" ? "lvlYou" : "lvlHer").style.width = pct + "%";
  $(which === "you" ? "numYou" : "numHer").textContent = db === -Infinity ? "-\u221e" : db.toFixed(0);
}
function appendLog(t){
  const el = $("log");
  if(el.textContent === "waiting for the bridge") el.textContent = "";
  el.textContent = (el.textContent + t).slice(-6000);
  el.scrollTop = el.scrollHeight;
}

let aws = null, actx = null, capNode = null, playNode = null, micStream = null;

function connectAudioWs(){
  const proto = location.protocol === "https:" ? "wss" : "ws";
  aws = new WebSocket(`${proto}://${location.host}${BASE}ws`);
  aws.binaryType = "arraybuffer";
  aws.onmessage = ev => {
    if(typeof ev.data !== "string"){
      const f = toF32(new Int16Array(ev.data));
      setLevel("her", f);
      if(playNode && actx) playNode.port.postMessage(resample(f, SR, actx.sampleRate));
      return;
    }
    const m = JSON.parse(ev.data);
    if(m.t === "log") appendLog(m.v);
    else if(m.t === "state"){
      if(m.state === "stopped" || m.state === "busy"){ setDialError(m.msg); endBrowserCall(true); }
      appendLog("\n[" + m.state + "] " + (m.msg || "") + "\n");
    }
  };
  aws.onclose = () => { if(S.browserCall) endBrowserCall(true); };
}

async function startBrowserCall(){
  setDialError("");
  try{
    micStream = await navigator.mediaDevices.getUserMedia({audio:{
      channelCount:1, echoCancellation:true, noiseSuppression:true, autoGainControl:true}});
  }catch(e){
    setDialError("Microphone access denied. Allow it in the browser and try again."); return;
  }
  if(!aws || aws.readyState !== 1){
    connectAudioWs();
    await new Promise(r => { const t = setInterval(() => {
      if(aws.readyState === 1){ clearInterval(t); r(); } }, 60); });
  }

  actx = new (window.AudioContext || window.webkitAudioContext)({sampleRate:SR, latencyHint:"interactive"});
  await actx.resume();
  await actx.audioWorklet.addModule(blobURL(CAP_SRC));
  await actx.audioWorklet.addModule(blobURL(PLAY_SRC));
  if(Math.abs(actx.sampleRate - SR) > 1)
    appendLog(`\n[ui] browser gave ${actx.sampleRate} Hz, resampling to 24000\n`);

  const src = actx.createMediaStreamSource(micStream);
  capNode = new AudioWorkletNode(actx, "cap", {numberOfInputs:1, numberOfOutputs:0});
  capNode.port.onmessage = e => {
    const f = resample(e.data, actx.sampleRate, SR);
    setLevel("you", f);
    if(aws && aws.readyState === 1) aws.send(toI16(f).buffer);
  };
  src.connect(capNode);

  playNode = new AudioWorkletNode(actx, "play", {numberOfInputs:0, numberOfOutputs:1, outputChannelCount:[1]});
  playNode.connect(actx.destination);

  aws.send(JSON.stringify({t:"start"}));
  S.browserCall = true; renderDialer(); setView("calls");
}

function endBrowserCall(silent){
  if(!silent && aws && aws.readyState === 1) aws.send(JSON.stringify({t:"stop"}));
  if(playNode){ playNode.port.postMessage("flush"); playNode.disconnect(); playNode = null; }
  if(capNode){ capNode.disconnect(); capNode = null; }
  if(micStream){ micStream.getTracks().forEach(t => t.stop()); micStream = null; }
  if(actx){ actx.close(); actx = null; }
  S.browserCall = false;
  setLevel("you", new Float32Array(1)); setLevel("her", new Float32Array(1));
  renderDialer();
}

document.addEventListener("keydown", e => {
  if(["TEXTAREA","INPUT","SELECT"].includes(e.target.tagName)) return;
  if(e.code === "Escape" && S.browserCall) endBrowserCall(false);
});

/* ================================================================== */
fetch(BASE + "api/meta").then(r => r.json()).then(m => {
  S.meta = m;
  $("twilioNumber").textContent = m.twilioNumber || "not configured";
  $("numberBox").classList.toggle("hide", !m.twilioNumber);
  if(S.config) fillConfig(S.config);
  renderDialer();
}).catch(() => {});
connectMonitor();
connectAudioWs();
render();
renderDialer();
</script>
</body>
</html>

Writing /kaggle/working/agent/index.html


## Cell 8 - Self-test

Starts the engine, feeds it 25 seconds of silence, and reports whether audio came back
and at what frame rate. Do this before opening a browser: a failure here is about SDL
and pipes, and a failure after it is about your microphone or the tunnel. Knowing which
saves a lot of time.

Silence is a fair test - she opens the conversation herself, so she should start
talking unprompted.

The 25 seconds start when the engine reports ready, so the first load of the 8.9 GB
q8_0 file - a couple of minutes off cold disk, seconds once it's in the page cache -
doesn't eat the budget.

In [8]:
import pathlib
pathlib.Path(f"{APP}/prompt.txt").write_text(PROMPT)

!cd {APP} && timeout 900 python bridge.py --binary "{PPX}" --workdir "{BINDIR}" \
    --model "{MODEL_DIR}" --warm-context {CONTEXT} \
    --selftest 25 --voice NATF1 --rate 1.0 --prompt "$(cat {APP}/prompt.txt)" 2>&1 | tail -30

ggml_cuda_init: found 2 CUDA devices:
  Device 0: Tesla T4, compute capability 7.5, VMM: yes
  Device 1: Tesla T4, compute capability 7.5, VMM: yes
load_backend: loaded CUDA backend from /kaggle/temp/pplex-runtime/moshi-bin-linux-x64-v0.8.0-beta/libggml-cuda.so
load_backend: loaded CPU backend from /kaggle/temp/pplex-runtime/moshi-bin-linux-x64-v0.8.0-beta/libggml-cpu-skylakex.so
using device: "CUDA0"
free memory: 14806 MiB
found model path: Codes4Fun/personaplex-7b-q8-GGUF/
seed: 1787077185
loading...
done loading. 5.337000
using voice embedding: Codes4Fun/personaplex-7b-q8-GGUF/voices/NATF1.gguf
[bridge 18:19:52] [selftest] mic pipe connected
CRITICAL: You are using the SDL disk i/o audio driver!
CRITICAL:  Reading from file [/kaggle/temp/pplex-runtime/moshi-bin-linux-x64-v0.8.0-beta/_pipes/selftest/mic.f32].
CRITICAL: You are using the SDL disk i/o audio driver!
CRITICAL:  Writing to file [/kaggle/temp/pplex-runtime/moshi-bin-linux-x64-v0.8.0-beta/_pipes/selftest/spk.f32].
engine re

## Cell 9 - Start the server

`--keep-warm` loads the model now and leaves it resident, so the first call dials
immediately instead of waiting on a cold start - which matters more with q8_0 than it
did with q4_k. The port only opens once the weights are in, so a first run after the
download can sit here for a minute or two.

`--max-engines` is the one new flag that needs a decision. Every concurrent call is
its own `personaplex` process, and each process is a full copy of the weights in
VRAM. On one T4 with q8_0 that means **one**: leave it at 1 and conference mode stays
greyed out. Two q4_k engines with a small context will fit; two q8_0 engines will not.

In [9]:
import subprocess, os, time, socket

subprocess.run("pkill -f bridge.py", shell=True)
subprocess.run("pkill -f personaplex", shell=True)
time.sleep(2)

# One engine per concurrent call, and one full copy of the weights per engine.
# q8_0 on a single T4 fits exactly one. Raise this only on a bigger card.
MAX_ENGINES = 1

cmd = ["python", "-u", f"{APP}/bridge.py",
       "--binary", PPX, "--workdir", BINDIR,
       "--model", MODEL_DIR,
       "--ui", f"{APP}/index.html",
       "--host", "0.0.0.0", "--port", str(APP_PORT),
       "--rate", "1.0", "--warm-context", str(CONTEXT),
       "--max-engines", str(MAX_ENGINES),
       "--keep-warm", "--warm-prompt", PROMPT]
if os.environ.get("PARTY_B_PHONE"):
    cmd += ["--party-b", os.environ["PARTY_B_PHONE"]]

server = subprocess.Popen(cmd, cwd=APP, env=dict(os.environ),
                          stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

# --keep-warm loads the weights before the socket opens, and 8.9 GB off a cold
# disk is not five seconds, so this waits longer than the q4 version did.
for i in range(420):
    if server.poll() is not None:
        print("server died, exit", server.returncode); break
    with socket.socket() as s:
        if s.connect_ex(("127.0.0.1", APP_PORT)) == 0:
            print(f"listening after {i}s"); break
    if i and i % 30 == 0:
        print(f"still loading the model... {i}s")
    time.sleep(1)
else:
    print("timed out - check the log below")

!tail -n 15 {LOG}

listening after 7s
load_backend: loaded CUDA backend from /kaggle/temp/pplex-runtime/moshi-bin-linux-x64-v0.8.0-beta/libggml-cuda.so
load_backend: loaded CPU backend from /kaggle/temp/pplex-runtime/moshi-bin-linux-x64-v0.8.0-beta/libggml-cpu-skylakex.so
using device: "CUDA0"
free memory: 14806 MiB
found model path: Codes4Fun/personaplex-7b-q8-GGUF/
seed: 1787077230
loading...
done loading. 5.707000
using voice embedding: Codes4Fun/personaplex-7b-q8-GGUF/voices/NATF1.gguf
[bridge 18:20:36] [engine0] mic pipe connected
CRITICAL: You are using the SDL disk i/o audio driver!
CRITICAL:  Reading from file [/kaggle/temp/pplex-runtime/moshi-bin-linux-x64-v0.8.0-beta/_pipes/engine0/mic.f32].
CRITICAL: You are using the SDL disk i/o audio driver!
CRITICAL:  Writing to file [/kaggle/temp/pplex-runtime/moshi-bin-linux-x64-v0.8.0-beta/_pipes/engine0/spk.f32].
[bridge 18:20:37] model warm


## Cell 10 - The link

Twilio dials back into this same URL for the audio stream, so the tunnel has to be up
before you place a call. It is - the server is already running behind it.

This cell also prints the two webhook URLs. You only need to paste them into the
Twilio console if you want **inbound** calls or the **browser-via-Twilio** path;
*Ring my phone* and conference dialling send their TwiML inline and need no console
setup at all.

In [10]:
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()
URL = ngrok.connect(APP_PORT, "http").public_url
WSS = URL.replace("https://", "wss://")

print("\n" + "=" * 68)
print("  Dashboard:", URL)
print("=" * 68)
print("""
Browser call : Call -> Browser (direct), press Connect. Wear headphones.
Phone call   : Call -> Ring my phone, answer, talk.
Inbound call : paste the voice webhook below into the Twilio console first.
""")
print("Twilio console, only if you want inbound or browser-via-Twilio:")
print(f"  Phone number -> Voice -> A call comes in (webhook, POST)")
print(f"      {URL}/voice/inbound")
print(f"  TwiML App -> Voice Request URL (POST)   [browser-via-Twilio only]")
print(f"      {URL}/voice/webrtc")
print(f"\n  Media stream Twilio connects back to: {WSS}/media-stream")
print(f"  Monitor socket the dashboard listens on: {WSS}/ws/monitor")

                                                                                                    
  Dashboard: https://3bac-34-87-104-222.ngrok-free.app

Browser call : Call -> Browser (direct), press Connect. Wear headphones.
Phone call   : Call -> Ring my phone, answer, talk.
Inbound call : paste the voice webhook below into the Twilio console first.

Twilio console, only if you want inbound or browser-via-Twilio:
  Phone number -> Voice -> A call comes in (webhook, POST)
      https://3bac-34-87-104-222.ngrok-free.app/voice/inbound
  TwiML App -> Voice Request URL (POST)   [browser-via-Twilio only]
      https://3bac-34-87-104-222.ngrok-free.app/voice/webrtc

  Media stream Twilio connects back to: wss://3bac-34-87-104-222.ngrok-free.app/media-stream
  Monitor socket the dashboard listens on: wss://3bac-34-87-104-222.ngrok-free.app/ws/monitor


## Cell 11 - Using it

Open the link and click through ngrok's interstitial. Three tabs across the top:
**Calls** is the live list and the transcript, **Call** places one, **Config** is the
persona.

### Four ways in

**Browser (direct).** *Call -> Browser (direct) -> Connect.* Your mic goes to the
model at 24 kHz over `/ws`, which is the notebook's own path and still the best
sounding one: no telephony codec, no Twilio minutes, no extra credentials. Wear
headphones - the model is full duplex and will answer its own voice coming out of
your speakers.

**Ring my phone.** *Call -> Ring my phone -> Connect.* Twilio dials
`TWILIO_TO_NUMBER` and connects it to `/media-stream`. The TwiML goes inline with the
REST call, so nothing needs configuring in the console.

**Inbound.** Point your Twilio number's voice webhook at `/voice/inbound` (Cell 10
prints the URL) and just call the number. This is the repo's main flow and it did not
exist in the old bridge.

**Browser via Twilio.** *Call -> Browser via Twilio.* A WebRTC leg out to Twilio and
straight back in as a Media Stream - the same 8 kHz mu-law path a real phone takes,
which makes it the honest way to hear what callers hear from a laptop. Needs the API
key pair and a TwiML App pointed at `/voice/webrtc`.

### Conference mode

Two legs, each with its own engine, cross-wired: caller A's translated audio is
written to caller B's stream and vice versa, which is `setOutputTarget` in the repo.
Set *Config -> Mode -> Conference*, fill in the party B phone, and the second leg gets
dialled automatically when the first one connects.

**This needs `MAX_ENGINES = 2` in Cell 9, and a card with room for two full copies of
the weights.** One T4 with q8_0 has room for one, so the dashboard blocks the button
and says so. Two q4_k engines at a low context do fit.

### The dashboard

The sidebar lists active and recent calls; clicking one shows its metrics and
transcript. Uptime, latency, jitter and packet counts are the repo's four; frame rate
and context-left are this notebook's, because they are the two numbers that tell you
whether a T4 is keeping up.

Only PersonaPlex's own speech is transcribed - the model writes its own words, not
yours. A caller turn shows up as *"Audio input detected"*, from an RMS gate, exactly
as in the repo: there is no ASR on the caller's side of this pipeline.

**Config** auto-saves as you type, 300 ms after the last keystroke. Two settings are
live and reach a call already in progress: **speaking pace**, which re-times audio
still in flight, and **band extension**, which is worth toggling mid-call because
whether it helps depends on the handset. Everything else - prompt, voice, temperature,
seed, context - is baked into the process at launch and applies to the **next** call,
because moshi.cpp holds the whole conversation inside the process and the only way to
clear it is a new one. Picking a *target language* overwrites the prompt with the
repo's interpreter template - choose a persona preset afterwards to get back to a
sales agent.

### Conversation length

This is the one hard limit worth knowing about. moshi.cpp has no sliding context
window: it consumes one frame every 80 ms whether anyone is speaking or not, so `-c`
fixes the maximum length of a conversation at start time. 3000 frames is the model's
maximum and buys **4 minutes**. With q8_0 that's roughly 11.5 GB of the T4's 14.8.

Past the limit the model doesn't stop, it degrades into nonsense, so the bridge
watches it: *Ctx left* counts down, an amber strip appears above the metrics at about
thirty seconds to go, and at zero the call is ended rather than left babbling. On a
phone call that means closing the media stream - with nothing after `<Connect>` in the
TwiML, Twilio ends the call itself.

## Cell 12 - Keep alive

Leave this running while you talk. It polls `/api/calls` - the repo's own endpoint -
so you get the same call list the dashboard shows, plus the engine's own output, which
is where the model's transcript and any CUDA complaints land. Interrupt the kernel when
you're done.

In [ ]:
import time, json, urllib.request

def calls():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{APP_PORT}/api/calls", timeout=3) as r:
            return json.load(r)["calls"]
    except Exception as e:
        return [{"callSid": f"(api unreachable: {e})", "status": "-", "mode": "-",
                 "duration": 0, "metrics": {}}]

while True:
    live = [c for c in calls() if c["status"] == "active"]
    if live:
        for c in live:
            m = c.get("metrics", {})
            print(f"{c['callSid']:<36} {c['mode']:<10} {c['duration']:6.0f}s  "
                  f"fps={m.get('fps', 0):5.2f}  lat={m.get('latency', 0):4.0f}ms  "
                  f"ctx={m.get('ctxLeft', 0):5.0f}s left")
    else:
        print("no active calls")
    !tail -n 6 {LOG}
    print("-" * 78)
    time.sleep(20)

no active calls
[bridge 18:20:36] [engine0] mic pipe connected
CRITICAL: You are using the SDL disk i/o audio driver!
CRITICAL:  Reading from file [/kaggle/temp/pplex-runtime/moshi-bin-linux-x64-v0.8.0-beta/_pipes/engine0/mic.f32].
CRITICAL: You are using the SDL disk i/o audio driver!
CRITICAL:  Writing to file [/kaggle/temp/pplex-runtime/moshi-bin-linux-x64-v0.8.0-beta/_pipes/engine0/spk.f32].
[bridge 18:20:37] model warm
------------------------------------------------------------------------------
no active calls
[bridge 18:20:36] [engine0] mic pipe connected
CRITICAL: You are using the SDL disk i/o audio driver!
CRITICAL:  Reading from file [/kaggle/temp/pplex-runtime/moshi-bin-linux-x64-v0.8.0-beta/_pipes/engine0/mic.f32].
CRITICAL: You are using the SDL disk i/o audio driver!
CRITICAL:  Writing to file [/kaggle/temp/pplex-runtime/moshi-bin-linux-x64-v0.8.0-beta/_pipes/engine0/spk.f32].
[bridge 18:20:37] model warm
-----------------------------------------------------------------

## Cell 13 - Cleanup

Also removes the named pipes. There is one pair per engine now, under
`_pipes/engine0`, `_pipes/engine1` - `Engine.start` recreates them anyway, so leaving
them is harmless, but a clean tree makes a later `ldd` or `ls` easier to read.

In [ ]:
import shutil
from pyngrok import ngrok

ngrok.kill()
try:
    server.terminate(); server.wait(timeout=10)
except Exception as e:
    print(e)
!pkill -f bridge.py || true
!pkill -f personaplex || true

shutil.rmtree(f"{BINDIR}/_pipes", ignore_errors=True)
print("stopped")

---

## How the phone call works

A call arrives - inbound to your number, outbound from *Ring my phone*, or a WebRTC
leg from the browser - and every one of them answers with the same TwiML:

```xml
<Response>
  <Connect>
    <Stream url="wss://.../media-stream">
      <Parameter name="role" value="caller-a" />
      <Parameter name="conference" value="translate-123" />
    </Stream>
  </Connect>
</Response>
```

Those `<Parameter>` elements are not decoration. Twilio strips query strings off a
`<Stream>` url, so per-call values have to travel as parameters and arrive in the
`start` event's `customParameters` - the blog post's first deployment tip, and what
makes one websocket endpoint able to serve single calls and both sides of a
conference.

`/media-stream` then behaves like `src/index.ts`: on `start` it registers the call,
builds a `BridgeSession`, and for a conference joins it to the named conference and
dials the second leg. Twilio's 8 kHz mu-law goes up to 24 kHz float32 for the model;
what comes back goes down to 8 kHz mu-law in 160-byte frames. Every mutation emits a
monitor event, which is how the dashboard stays live without polling.

## Two places this port deliberately differs from the repo

**No Opus.** The repo wraps caller audio in Ogg Opus because its PersonaPlex sits on a
RunPod GPU behind a websocket that only accepts Ogg containers. Ours is a process on
this machine reading float32 off a pipe, so encoding to Opus and immediately decoding
it again would be a lossy round trip for nothing. `src/audio-buffer.ts` goes with it:
it exists to cut audio into the fixed 480-sample frames an Opus encoder needs, and
nothing here needs fixed frames.

**Different mu-law encoder.** The repo's `linearToMulaw` clips at
`MULAW_MAX = 0x1fff`, which is 8191 - a constant from the 13-bit reference
implementation, applied to a 16-bit sample without the `>> 2` that would make it
correct. Everything louder than about -12 dBFS saturates to the same code, so loud
speech comes out distorted. This notebook's numpy lookup tables are bit-exact against
the reference codec across all 65,536 inputs, and the test cell above proves it, so
they stay. If you port anything back the other way, that line is worth fixing.

Rate conversion also stays as the Kaiser-windowed filter this notebook already had
rather than the repo's two-tap linear interpolator - the filter puts 105 dB between
the speech band and the 4 kHz fold point, so nothing aliases on the way down.

## Troubleshooting

**"off - missing ..." in Cell 2.** It names exactly which secret. Add it under
Add-ons -> Secrets, rerun Cell 2, then restart from Cell 9 - the server reads the
environment once, at startup.

**Conference is greyed out.** Either `MAX_ENGINES` is 1 in Cell 9, or party B's phone
is empty in Config. The button's tooltip says which.

**Twilio error 21219.** The number isn't verified on your trial account. Verify it in
the console under Phone Numbers -> Verified Caller IDs.

**Twilio error 21210 or 21212.** `TWILIO_FROM_NUMBER` isn't a Twilio number you own,
or isn't voice-capable. It must be a number Twilio issued you, not your own mobile.

**Twilio error 20003, or a 403.** Wrong SID or auth token, or the account is suspended.

**The token request times out on browser-via-Twilio.** `TWILIO_API_KEY_SID`,
`TWILIO_API_KEY_SECRET` and `TWILIO_TWIML_APP_SID` all have to be present, and the
API key has to belong to the same account as the SID. The dashboard prints the
missing one.

**Inbound calls hit ngrok's interstitial.** Free ngrok shows a browser warning page on
HTML fetches, which can break a webhook. Websocket upgrades go straight through,
which is why *Ring my phone* and conference dialling send TwiML inline instead. If
`/voice/inbound` is the one failing, a paid ngrok domain or another tunnel fixes it.

**The phone rings but there's silence.** Look at Cell 12's output. `[MediaStream]
Client connected` means the audio path is up and the engine is the problem; no such
line means Twilio never reached the tunnel.

**"No engine available."** Something else already holds the only engine - a browser
call and a phone call can't share one, because moshi.cpp keeps one conversation per
process. Hang up the first one.

**Numbers rejected as invalid.** E.164 only: `+919876543210`, not `9876543210` or
`+91 98765 43210`.

**She talks over you in the browser.** Headphones. There is no second fix; the model is
full duplex and will respond to its own voice.

**Out of memory, or the engine dies partway through loading.** q8_0 needs about 9 GB
resident before any context is counted. Lower `CONTEXT` in Cell 5 - 2000 frames still
gives you two and a half minutes - then rerun Cells 8 and 9. To go back to the smaller
weights entirely, set `MODEL_DIR = "Codes4Fun/personaplex-7b-v1-q4_k-GGUF"` and
`MODEL_GGUF = "model-q4_k.gguf"` in Cell 5 and point the manifest at
`Codes4Fun/personaplex-7b-v1-q4_k-GGUF` on Hugging Face.

**"error: missing moshi file" or "missing mimi file".** The model directory is there but
something inside it isn't. Rerun Cell 5; it names what's absent. The mimi error in
particular means `Codes4Fun/moshi-common/` didn't come down - the q8 folder's config
reaches for it as `../moshi-common`.

**Cell 8 returns zero bytes.** The engine started but nothing came back through the
playback pipe. If it never printed `ready` it's a model or VRAM problem, not a pipe
problem, and `-c 800` is the first thing to try.

**No disk driver in Cell 3.** SDL's disk audio driver is a compile-time option. Without
it this approach can't work and you'd need a real audio device - PulseAudio with a null
sink and a pipe source is the usual substitute.

**Everything died at once.** The kernel restarted or the session hit its limit. Rerun
from Cell 2; the binary and weights are still on disk, so it's quick the second time.

Source repo for the call handling: https://github.com/chaosloth/spike-persona-plex